# STMA SMCI Complete Workflow


This notebook performs the complete STMA soil-moisture-content index workflow:

1. Extracts 50 historical monthly records from Google Earth Engine at 30 m.
2. Uses Prophet to forecast the 7 dynamic factors for January 2030 and January 2050.
3. Extracts/merges the 3 static factors.
4. Trains and validates XGBoost and MLP-ANN classifiers using the January 2024 labelled data.
5. Predicts January 2024, January 2030 and January 2050 soil-moisture classes.
6. Generates Pearson correlation, 5-fold verification, feature importance, SHAP explainability, SHAP-IQ optional output, and final natural-break maps.
7. Saves all outputs in Google Drive under `STMA_SMCI` with checkpoint/resume protection.


# Cell 1 — Install Required Packages

In [ ]:
# ============================================================
# CELL 1 — INSTALL REQUIRED PACKAGES
# ============================================================
# Run this cell first. It installs the libraries used for GEE, Prophet,
# XGBoost, MLP-ANN, SHAP, raster export, natural breaks and Excel output.

!pip install -q earthengine-api geemap prophet xgboost shap openpyxl joblib rasterio pyproj shapely jenkspy scikit-learn tensorflow

# SHAP-IQ is optional. The notebook will continue if installation fails.
!pip install -q shapiq || true


# Cell 2 — Import Libraries and Configure Plot Style

In [ ]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES AND CONFIGURE PLOT STYLE
# ============================================================
import os
import re
import json
import math
import time
import glob
import shutil
import warnings
import traceback
from pathlib import Path
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from google.colab import drive

import ee

from prophet import Prophet
from joblib import Parallel, delayed, dump, load

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score,
    roc_curve, auc, log_loss
)
from scipy.stats import randint, uniform

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

import shap

import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask as rio_mask
from pyproj import Transformer
from shapely.geometry import shape, mapping
from shapely.ops import transform as shapely_transform
import jenkspy

# ----------------------------
# Global figure style
# ----------------------------
PREFERRED_FONT = "Times New Roman"
FALLBACK_FONT = "Liberation Serif"

mpl.rcParams["font.family"] = [PREFERRED_FONT, FALLBACK_FONT, "serif"]
mpl.rcParams["axes.titlesize"] = 18
mpl.rcParams["axes.labelsize"] = 15
mpl.rcParams["xtick.labelsize"] = 12
mpl.rcParams["ytick.labelsize"] = 12
mpl.rcParams["legend.fontsize"] = 12
mpl.rcParams["figure.titlesize"] = 20
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)


# Cell 3 — Mount Google Drive and Create STMA_SMCI Folder Structure

In [ ]:
# ============================================================
# CELL 3 — MOUNT GOOGLE DRIVE AND CREATE STMA_SMCI FOLDER STRUCTURE
# ============================================================
drive.mount('/content/drive')

ROOT_DIR = Path('/content/drive/MyDrive/STMA_SMCI')
GEE_STAGING_DIR = Path('/content/drive/MyDrive/STMA_SMCI_GEE_EXPORTS')  # simple root-level staging folder for GEE Drive exports

DIRS = {
    "metadata": ROOT_DIR / "00_Project_Metadata",
    "historical": ROOT_DIR / "01_GEE_Historical_Extractions_30m",
    "training": ROOT_DIR / "02_January2024_Training_Data",
    "layers_2024": ROOT_DIR / "03_January2024_Factor_Layers",
    "prophet_input": ROOT_DIR / "04_Prophet_Input_Tables",
    "prophet_checkpoints": ROOT_DIR / "05_Prophet_Checkpoints",
    "prophet_metrics": ROOT_DIR / "06_Prophet_Validation_Metrics",
    "forecast_tables": ROOT_DIR / "07_Prophet_Forecast_Point_Tables",
    "layers_2030": ROOT_DIR / "08_Forecast_Factor_Layers_2030",
    "layers_2050": ROOT_DIR / "09_Forecast_Factor_Layers_2050",
    "xgb": ROOT_DIR / "10_XGBoost_Model_Outputs",
    "prediction_layers": ROOT_DIR / "11_XGBoost_MLP_Prediction_Layers",
    "figures": ROOT_DIR / "12_Explainability_Figures",
    "final": ROOT_DIR / "13_Final_Compiled_Outputs",
    "logs": ROOT_DIR / "14_Error_Logs",
    "gee_staging": GEE_STAGING_DIR
}

# Subfolders for figures
FIG_SUBDIRS = {
    "shap_dependence": DIRS["figures"] / "SHAP_Dependence_Plots",
    "waterfall": DIRS["figures"] / "SHAP_Waterfall_Plots",
    "shapiq": DIRS["figures"] / "SHAP_IQ_Outputs",
    "mlp": DIRS["figures"] / "MLP_ANN_Figures",
    "natural_breaks": DIRS["prediction_layers"] / "Natural_Breaks_Classified_Maps",
    "probability": DIRS["prediction_layers"] / "Probability_Maps",
    "scores": DIRS["prediction_layers"] / "SMCI_Score_Maps",
    "three_class": DIRS["prediction_layers"] / "Three_Class_Maps",
    "percentages": DIRS["prediction_layers"] / "Class_Area_Percentage_Tables"
}

for p in list(DIRS.values()) + list(FIG_SUBDIRS.values()):
    p.mkdir(parents=True, exist_ok=True)

print("Root folder:", ROOT_DIR)
print("GEE staging folder:", GEE_STAGING_DIR)
print("Folder structure is ready.")


# Cell 4 — Main Configuration and Resume-Safe Switches

In [ ]:
# ============================================================
# CELL 4 — MAIN CONFIGURATION AND RESUME-SAFE SWITCHES
# ============================================================
# The default switch values are False. This means the notebook will continue
# from existing Google Drive outputs rather than starting from fresh.

FORCE_RERUN_GEE_EXPORTS = False
FORCE_RETUNE_PROPHET = False
FORCE_RERUN_PROPHET_BATCHES = False
FORCE_RETRAIN_XGBOOST = False
FORCE_RETRAIN_MLP = False
FORCE_REMAKE_FIGURES = False
FORCE_RERASTERIZE = False

# If True, Colab waits for Earth Engine export tasks to finish.
# For long exports, you may set this False, let GEE run, and rerun the notebook later.
WAIT_FOR_GEE_EXPORTS = True
GEE_TASK_POLL_SECONDS = 60

# ----------------------------
# GEE asset paths
# ----------------------------
ASSET_TRAINING_POINTS = 'projects/askim-486112/assets/SMCI_2024JANGEE'
ASSET_STMA_BOUNDARY = 'projects/askim-486112/assets/STMA_Shapefile'
ASSET_TIMESERIES_POINTS = 'projects/askim-486112/assets/SMCI_CELL30'

# ----------------------------
# Spatial settings
# ----------------------------
CELL_SIZE_M = 30
OUTPUT_CRS = 'EPSG:32630'  # UTM Zone 30N, appropriate for western Ghana/STMA
GEE_EXPORT_DRIVE_FOLDER = 'STMA_SMCI_GEE_EXPORTS'  # Earth Engine exports to this simple Drive folder, then Colab moves files into STMA_SMCI subfolders.

# ----------------------------
# Historical selected months
# ----------------------------
SELECTED_MONTHS = [1, 3, 5, 7, 9, 11]
MONTH_NAMES = {
    1: 'JANUARY', 3: 'MARCH', 5: 'MAY', 7: 'JULY', 9: 'SEPTEMBER', 11: 'NOVEMBER'
}

# Full years 2018-2025 plus January and March 2026 only.
HISTORICAL_YEAR_MONTHS = []
for y in range(2018, 2026):
    for m in SELECTED_MONTHS:
        HISTORICAL_YEAR_MONTHS.append((y, m))
HISTORICAL_YEAR_MONTHS += [(2026, 1), (2026, 3)]

EXPECTED_HISTORICAL_FILE_COUNT = len(HISTORICAL_YEAR_MONTHS)
VALIDATION_STEPS = 6

# ----------------------------
# Predictor factors
# ----------------------------
DYNAMIC_FACTORS = ['NDMI', 'EVI', 'MSAVI', 'AWEIsh', 'BSI', 'NDBI', 'Rainfall']
STATIC_FACTORS = ['Elevation', 'Slope', 'TWI']
FEATURE_COLUMNS = DYNAMIC_FACTORS + STATIC_FACTORS

# Target logic
TARGET_CANDIDATES = ['Target', 'BC']
CLASS_LABELS = {
    0: 'Low moisture zones',
    1: 'Moderate moisture zones',
    2: 'Moist zones'
}
NB_CLASS_LABELS = {
    1: 'Very Low',
    2: 'Low',
    3: 'Moderate',
    4: 'High',
    5: 'Very High'
}

# Prophet settings
FORECAST_TARGET_DATES = [pd.Timestamp('2030-01-01'), pd.Timestamp('2050-01-01')]
PROPHET_BATCH_SIZE_POINTS = 200        # Smaller = safer checkpoints. Increase if Colab is stable.
N_JOBS_PROPHET = max(1, min(2, os.cpu_count() or 1))  # Prophet can be heavy. Keep modest for stability.
MAX_PROPHET_BATCHES_PER_RUN = None     # Set to an integer if you want to limit each Colab run.

CHANGEPOINT_PRIOR_OPTIONS = [0.01, 0.05, 0.10]
SEASONALITY_PRIOR_OPTIONS = [5.0, 10.0, 15.0]
YEARLY_FOURIER_ORDER_OPTIONS = [3, 5]

# XGBoost settings
XGB_RANDOM_SEARCH_ITERATIONS = 20
XGB_CV_FOLDS = 5
TEST_SIZE = 0.30

# MLP-ANN settings
MLP_EPOCHS = 250
MLP_BATCH_SIZE = 64
MLP_PATIENCE = 30

# SHAP settings
SHAP_SAMPLE_SIZE = 1500
SHAP_INTERACTION_SAMPLE_SIZE = 800
SHAP_IQ_SAMPLE_SIZE = 100

# Value clamps for forecasted dynamic factors
CLAMP_RULES = {
    'NDMI': (-1, 1),
    'EVI': (-1, 1),
    'MSAVI': (-1, 1),
    'AWEIsh': (-3, 3),
    'BSI': (-1, 1),
    'NDBI': (-1, 1),
    'Rainfall': (0, None)
}

settings = {
    "ASSET_TRAINING_POINTS": ASSET_TRAINING_POINTS,
    "ASSET_STMA_BOUNDARY": ASSET_STMA_BOUNDARY,
    "ASSET_TIMESERIES_POINTS": ASSET_TIMESERIES_POINTS,
    "CELL_SIZE_M": CELL_SIZE_M,
    "OUTPUT_CRS": OUTPUT_CRS,
    "HISTORICAL_YEAR_MONTHS": HISTORICAL_YEAR_MONTHS,
    "DYNAMIC_FACTORS": DYNAMIC_FACTORS,
    "STATIC_FACTORS": STATIC_FACTORS,
    "FEATURE_COLUMNS": FEATURE_COLUMNS,
    "VALIDATION_STEPS": VALIDATION_STEPS
}

with open(DIRS["metadata"] / "workflow_settings.json", "w") as f:
    json.dump(settings, f, indent=2, default=str)

with open(DIRS["metadata"] / "asset_paths.txt", "w") as f:
    f.write(f"Training points: {ASSET_TRAINING_POINTS}\n")
    f.write(f"STMA boundary: {ASSET_STMA_BOUNDARY}\n")
    f.write(f"Time-series points: {ASSET_TIMESERIES_POINTS}\n")

print("Configuration loaded.")
print("Expected historical records:", EXPECTED_HISTORICAL_FILE_COUNT)
print("Dynamic factors:", DYNAMIC_FACTORS)
print("Static factors:", STATIC_FACTORS)


# Cell 5 — Utility Functions for Logging, Checking and File Safety

In [ ]:
# ============================================================
# CELL 5 — UTILITY FUNCTIONS FOR LOGGING, CHECKING AND FILE SAFETY
# ============================================================
def now_string():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def log_message(message, log_name='run_log.txt'):
    log_path = DIRS['metadata'] / log_name
    line = f"[{now_string()}] {message}"
    print(line)
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(line + '\n')


def log_error(stage, error):
    err_path = DIRS['logs'] / f"{stage}_errors.txt"
    with open(err_path, 'a', encoding='utf-8') as f:
        f.write(f"\n[{now_string()}] {stage}\n")
        f.write(str(error) + "\n")
        f.write(traceback.format_exc() + "\n")
    print(f"Error logged to {err_path}")


def csv_has_columns(path, required_cols, min_rows=1):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        sample = pd.read_csv(path, nrows=max(min_rows, 5))
        sample.columns = sample.columns.str.strip()
        if min_rows > 0 and sample.shape[0] < min_rows:
            return False
        return all(c in sample.columns for c in required_cols)
    except Exception:
        return False


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=str)


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def safe_read_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def clean_numeric_features(df, feature_cols, medians=None):
    out = df.copy()
    for c in feature_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    if medians is None:
        medians = out[feature_cols].median(numeric_only=True)
    for c in feature_cols:
        fill_val = medians[c] if c in medians.index and pd.notna(medians[c]) else 0
        out[c] = out[c].fillna(fill_val)
    return out, medians


def apply_clamps(df, factor_cols=DYNAMIC_FACTORS):
    out = df.copy()
    for c in factor_cols:
        if c in out.columns and c in CLAMP_RULES:
            low, high = CLAMP_RULES[c]
            if low is not None:
                out[c] = out[c].clip(lower=low)
            if high is not None:
                out[c] = out[c].clip(upper=high)
    return out


def month_label(year, month):
    return f"{MONTH_NAMES[month]}_{year}"


def date_from_year_month(year, month):
    return pd.Timestamp(year=int(year), month=int(month), day=1)

log_message("Utility functions loaded.")


# Cell 6 — Authenticate and Initialise Google Earth Engine

In [ ]:
# ============================================================
# CELL 6 — AUTHENTICATE AND INITIALISE GOOGLE EARTH ENGINE
# ============================================================
try:
    ee.Initialize(project='askim-486112')
    log_message("Earth Engine initialised without new authentication.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project='askim-486112')
    log_message("Earth Engine authenticated and initialised.")

# Load key assets
training_points_raw = ee.FeatureCollection(ASSET_TRAINING_POINTS)
stma_boundary = ee.FeatureCollection(ASSET_STMA_BOUNDARY)
timeseries_points_raw = ee.FeatureCollection(ASSET_TIMESERIES_POINTS)

print("Training point count:", training_points_raw.size().getInfo())
print("Time-series point count:", timeseries_points_raw.size().getInfo())
print("STMA boundary feature count:", stma_boundary.size().getInfo())


# Cell 7 — Earth Engine Helper Functions for Geometry, Factors and Sampling

In [ ]:
# ============================================================
# CELL 7 — EARTH ENGINE HELPER FUNCTIONS FOR GEOMETRY, FACTORS AND SAMPLING
# ============================================================
# Coordinate convention used here follows the project notes:
# X = Latitude and Y = Longitude. Geometry is Point([Y, X]).

S2_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
S2_BACKUP_START = '2018-01-01'
S2_BACKUP_END = '2026-04-01'
CHIRPS_BACKUP_START_YEAR = 2018
CHIRPS_BACKUP_END_YEAR = 2025


def standardize_points(fc, include_target=False):
    """Standardise OBJECTID, X, Y and optional Target/BC to Target, and force point geometry from X/Y."""
    def _one(f):
        names = f.propertyNames()
        obj = ee.Algorithms.If(names.contains('OBJECTID'), f.get('OBJECTID'),
              ee.Algorithms.If(names.contains('ObjectID'), f.get('ObjectID'),
              ee.Algorithms.If(names.contains('objectid'), f.get('objectid'),
              ee.Algorithms.If(names.contains('OBJECTID *'), f.get('OBJECTID *'),
              ee.Algorithms.If(names.contains('OID'), f.get('OID'),
              ee.Algorithms.If(names.contains('OID '), f.get('OID '), f.id()))))))

        # Use X/Y properties to define geometry. X = latitude, Y = longitude.
        lat = ee.Number.parse(ee.String(f.get('X')))
        lon = ee.Number.parse(ee.String(f.get('Y')))
        geom = ee.Geometry.Point([lon, lat], 'EPSG:4326')
        out = ee.Feature(geom).set({'OBJECTID': obj, 'X': lat, 'Y': lon})

        if include_target:
            target = ee.Algorithms.If(names.contains('Target'), f.get('Target'),
                     ee.Algorithms.If(names.contains('BC'), f.get('BC'), None))
            out = out.set('Target', target)
        return out
    return fc.map(_one)


def s2_collection(start_date, end_date, region, mask_clouds=True):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(region)
           .filterDate(start_date, end_date)
           .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 100)))

    def _prep(img):
        scaled = img.select(S2_BANDS).multiply(0.0001).toFloat()
        if mask_clouds:
            scl = img.select('SCL')
            mask = (scl.neq(3)
                    .And(scl.neq(8))
                    .And(scl.neq(9))
                    .And(scl.neq(10))
                    .And(scl.neq(11)))
            scaled = scaled.updateMask(mask)
        return scaled.copyProperties(img, img.propertyNames())
    return col.map(_prep)


def median_or_constant(col, band_names, fill_value=0):
    constant = ee.Image.constant([fill_value] * len(band_names)).rename(band_names).toFloat()
    return ee.Image(ee.Algorithms.If(col.size().gt(0), col.median().toFloat(), constant))


def compute_s2_indices(s2_img):
    blue = s2_img.select('B2')
    green = s2_img.select('B3')
    red = s2_img.select('B4')
    nir = s2_img.select('B8')
    swir1 = s2_img.select('B11')
    swir2 = s2_img.select('B12')

    ndmi = nir.subtract(swir1).divide(nir.add(swir1).add(0.0001)).rename('NDMI').clamp(-1, 1)

    evi = (nir.subtract(red).multiply(2.5)
           .divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1).add(0.0001))
           .rename('EVI').clamp(-1, 1))

    msavi_term = nir.multiply(2).add(1)
    msavi = (msavi_term.subtract(
        msavi_term.pow(2).subtract(nir.subtract(red).multiply(8)).max(0).sqrt()
    ).divide(2).rename('MSAVI').clamp(-1, 1))

    aweish = (blue.add(green.multiply(2.5))
              .subtract(nir.add(swir1).multiply(1.5))
              .subtract(swir2.multiply(0.25))
              .rename('AWEIsh'))

    bsi = (swir1.add(red).subtract(nir.add(blue))
           .divide(swir1.add(red).add(nir).add(blue).add(0.0001))
           .rename('BSI').clamp(-1, 1))

    ndbi = swir1.subtract(nir).divide(swir1.add(nir).add(0.0001)).rename('NDBI').clamp(-1, 1)

    return ee.Image.cat([ndmi, evi, msavi, aweish, bsi, ndbi]).toFloat()


def s2_index_image_for_month(year, month, region):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    target_masked = compute_s2_indices(median_or_constant(s2_collection(start, end, region, True), S2_BANDS, 0))
    target_raw = compute_s2_indices(median_or_constant(s2_collection(start, end, region, False), S2_BANDS, 0))

    same_month_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                      .filterBounds(region)
                      .filterDate(S2_BACKUP_START, S2_BACKUP_END)
                      .filter(ee.Filter.calendarRange(month, month, 'month'))
                      .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 100)))

    def _prep_backup(img):
        scaled = img.select(S2_BANDS).multiply(0.0001).toFloat()
        scl = img.select('SCL')
        mask = (scl.neq(3)
                .And(scl.neq(8))
                .And(scl.neq(9))
                .And(scl.neq(10))
                .And(scl.neq(11)))
        return scaled.updateMask(mask)

    same_month = compute_s2_indices(median_or_constant(same_month_col.map(_prep_backup), S2_BANDS, 0))

    longterm_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                    .filterBounds(region)
                    .filterDate(S2_BACKUP_START, S2_BACKUP_END)
                    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 100))
                    .map(_prep_backup))
    longterm = compute_s2_indices(median_or_constant(longterm_col, S2_BANDS, 0))

    fallback_constants = ee.Image.constant([0, 0, 0, 0, 0, 0]).rename(['NDMI', 'EVI', 'MSAVI', 'AWEIsh', 'BSI', 'NDBI']).toFloat()

    return (target_masked
            .unmask(target_raw)
            .unmask(same_month)
            .unmask(longterm)
            .unmask(fallback_constants)
            .toFloat())


def chirps_monthly_sum(year, month, region):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    col = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
           .filterBounds(region)
           .filterDate(start, end)
           .select('precipitation'))
    zero = ee.Image.constant(0).rename('Rainfall').toFloat()
    img = ee.Image(ee.Algorithms.If(col.size().gt(0), col.sum().rename('Rainfall').toFloat(), zero))
    return img


def rainfall_image_for_month(year, month, region):
    target = chirps_monthly_sum(year, month, region)

    same_month_sums = []
    for yy in range(CHIRPS_BACKUP_START_YEAR, CHIRPS_BACKUP_END_YEAR + 1):
        same_month_sums.append(chirps_monthly_sum(yy, month, region))
    same_month_mean = ee.ImageCollection.fromImages(same_month_sums).mean().rename('Rainfall').toFloat()

    all_sums = []
    for yy in range(CHIRPS_BACKUP_START_YEAR, CHIRPS_BACKUP_END_YEAR + 1):
        for mm in SELECTED_MONTHS:
            all_sums.append(chirps_monthly_sum(yy, mm, region))
    longterm_mean = ee.ImageCollection.fromImages(all_sums).mean().rename('Rainfall').toFloat()

    return target.unmask(same_month_mean).unmask(longterm_mean).unmask(ee.Image.constant(0).rename('Rainfall')).toFloat()


def static_factor_image(region):
    dem = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('Elevation').toFloat()
    slope = ee.Terrain.slope(dem).rename('Slope').toFloat()

    merit = ee.Image('MERIT/Hydro/v1_0_1')
    flow_acc_km2 = merit.select('upa')
    specific_catchment_area = flow_acc_km2.multiply(1000000).divide(CELL_SIZE_M).max(500)
    tan_slope = slope.multiply(math.pi).divide(180).tan().max(0.001)
    twi = specific_catchment_area.divide(tan_slope).log().rename('TWI').toFloat()

    static = ee.Image.cat([dem, slope, twi]).toFloat()

    # Fill rare voids using focal mean and final constants.
    static_filled = static.unmask(static.focal_mean(radius=90, units='meters'))
    static_filled = static_filled.unmask(ee.Image.constant([0, 0, 10]).rename(STATIC_FACTORS).toFloat())
    return static_filled


def dynamic_factor_image(year, month, region):
    s2idx = s2_index_image_for_month(year, month, region)
    rainfall = rainfall_image_for_month(year, month, region)
    return ee.Image.cat([s2idx, rainfall]).select(DYNAMIC_FACTORS).toFloat()


def ten_factor_image(year, month, region):
    return ee.Image.cat([dynamic_factor_image(year, month, region), static_factor_image(region)]).select(FEATURE_COLUMNS).toFloat()


def add_date_properties(fc, year, month):
    label = month_label(year, month)
    def _set(f):
        return f.set({'Year': year, 'Month': month, 'Date_Label': label})
    return fc.map(_set)

log_message("Earth Engine helper functions loaded.")


# Cell 8 — Standardise Point Assets and Define Processing Region

In [ ]:
# ============================================================
# CELL 8 — STANDARDISE POINT ASSETS AND DEFINE PROCESSING REGION
# ============================================================
# No boundary clipping is used for time-series extraction. A buffered extent around
# the point grid is used to protect boundary points and avoid missing values.

timeseries_points = standardize_points(timeseries_points_raw, include_target=False)
training_points = standardize_points(training_points_raw, include_target=True)

processing_region = timeseries_points.geometry().bounds().buffer(5000)
training_region = training_points.geometry().bounds().buffer(5000)

projection_30m = ee.Projection(OUTPUT_CRS).atScale(CELL_SIZE_M)

print("Standardised time-series point count:", timeseries_points.size().getInfo())
print("Standardised training point count:", training_points.size().getInfo())
print("Processing region prepared without boundary clipping.")


# Cell 9 — Earth Engine Export Helpers and Staging Sync

In [ ]:
# ============================================================
# CELL 9 — EARTH ENGINE EXPORT HELPERS AND STAGING SYNC
# ============================================================
def monitor_ee_tasks(tasks, poll_seconds=60):
    if not tasks:
        print("No new Earth Engine tasks to monitor.")
        return
    unfinished = list(tasks)
    while unfinished:
        states = []
        for task in unfinished:
            status = task.status()
            states.append(status.get('state'))
        state_counts = pd.Series(states).value_counts().to_dict()
        print(f"{now_string()} | Task states: {state_counts}")

        still_running = []
        for task in unfinished:
            status = task.status()
            state = status.get('state')
            if state in ['COMPLETED']:
                print("Completed:", status.get('description'))
            elif state in ['FAILED', 'CANCELLED']:
                print("Task issue:", status)
                with open(DIRS['logs'] / 'gee_export_errors.csv', 'a', encoding='utf-8') as f:
                    f.write(json.dumps(status) + '\n')
            else:
                still_running.append(task)
        unfinished = still_running
        if unfinished:
            time.sleep(poll_seconds)


def move_staged_csvs_to(target_dir):
    """Move CSV files from the GEE staging folder into the intended project subfolder."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    for p in GEE_STAGING_DIR.glob('*.csv'):
        dest = target_dir / p.name
        if not dest.exists():
            shutil.move(str(p), str(dest))
            print("Moved:", p.name, "→", dest)


def move_staged_tifs_to(target_dir, prefix_filter=None):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    patterns = ['*.tif', '*.tiff', '*.TFRecord*']
    files = []
    for pat in patterns:
        files += list(GEE_STAGING_DIR.glob(pat))
    for p in files:
        if prefix_filter and not p.name.startswith(prefix_filter):
            continue
        dest = target_dir / p.name
        if not dest.exists():
            shutil.move(str(p), str(dest))
            print("Moved:", p.name, "→", dest)


def export_table_to_drive_if_needed(fc, description, selectors, destination_dir, force=False):
    destination_dir = Path(destination_dir)
    out_csv = destination_dir / f"{description}.csv"
    required_cols = selectors
    if out_csv.exists() and csv_has_columns(out_csv, required_cols, min_rows=1) and not force:
        log_message(f"Skipping existing CSV: {out_csv.name}")
        return None

    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=description,
        folder=GEE_EXPORT_DRIVE_FOLDER,
        fileNamePrefix=description,
        fileFormat='CSV',
        selectors=selectors
    )
    task.start()
    log_message(f"Submitted GEE table export: {description}")
    return task

log_message("GEE export helpers ready.")


# Cell 10 — Submit Historical 30 m Dynamic-Factor Exports from GEE

In [ ]:
# ============================================================
# CELL 10 — SUBMIT HISTORICAL 30 m DYNAMIC-FACTOR EXPORTS FROM GEE
# ============================================================
# Exports 50 CSV files:
# January, March, May, July, September and November from 2018–2025,
# plus January and March 2026.

historical_tasks = []
hist_selectors = ['OBJECTID', 'X', 'Y', 'Year', 'Month', 'Date_Label'] + DYNAMIC_FACTORS

for year, month in HISTORICAL_YEAR_MONTHS:
    label = month_label(year, month)
    out_path = DIRS['historical'] / f"{label}.csv"

    if out_path.exists() and csv_has_columns(out_path, hist_selectors, min_rows=1) and not FORCE_RERUN_GEE_EXPORTS:
        log_message(f"Historical extraction already exists and is valid: {label}.csv")
        continue

    try:
        img = dynamic_factor_image(year, month, processing_region)
        sampled = img.sampleRegions(
            collection=timeseries_points,
            properties=['OBJECTID', 'X', 'Y'],
            scale=CELL_SIZE_M,
            projection=projection_30m,
            geometries=False,
            tileScale=16
        )
        sampled = add_date_properties(sampled, year, month)
        task = export_table_to_drive_if_needed(sampled, label, hist_selectors, DIRS['historical'], FORCE_RERUN_GEE_EXPORTS)
        if task is not None:
            historical_tasks.append(task)
    except Exception as e:
        log_error(f"GEE_Historical_{label}", e)

print("Submitted new historical tasks:", len(historical_tasks))
if WAIT_FOR_GEE_EXPORTS:
    monitor_ee_tasks(historical_tasks, poll_seconds=GEE_TASK_POLL_SECONDS)
    move_staged_csvs_to(DIRS['historical'])
else:
    print("WAIT_FOR_GEE_EXPORTS is False. Let tasks finish, then rerun this and the next cells.")


# Cell 11 — Export Static Factors for the Full 30 m Point Grid

In [ ]:
# ============================================================
# CELL 11 — EXPORT STATIC FACTORS FOR THE FULL 30 m POINT GRID
# ============================================================
static_selectors = ['OBJECTID', 'X', 'Y'] + STATIC_FACTORS
static_path = DIRS['prophet_input'] / 'SMCI_CELL30_Static_Factors_30m.csv'
static_tasks = []

if static_path.exists() and csv_has_columns(static_path, static_selectors, min_rows=1) and not FORCE_RERUN_GEE_EXPORTS:
    log_message("Static factors already exist and are valid.")
else:
    try:
        static_img = static_factor_image(processing_region)
        sampled_static = static_img.sampleRegions(
            collection=timeseries_points,
            properties=['OBJECTID', 'X', 'Y'],
            scale=CELL_SIZE_M,
            projection=projection_30m,
            geometries=False,
            tileScale=16
        )
        task = export_table_to_drive_if_needed(sampled_static, 'SMCI_CELL30_Static_Factors_30m', static_selectors, DIRS['prophet_input'], FORCE_RERUN_GEE_EXPORTS)
        if task is not None:
            static_tasks.append(task)
    except Exception as e:
        log_error("GEE_Static_Factors", e)

if WAIT_FOR_GEE_EXPORTS:
    monitor_ee_tasks(static_tasks, poll_seconds=GEE_TASK_POLL_SECONDS)
    move_staged_csvs_to(DIRS['prophet_input'])
else:
    print("WAIT_FOR_GEE_EXPORTS is False. Let static export finish, then rerun this cell.")


# Cell 12 — Export January 2024 Training Predictors with Target/BC

In [ ]:
# ============================================================
# CELL 12 — EXPORT JANUARY 2024 TRAINING PREDICTORS WITH TARGET/BC
# ============================================================
training_selectors = ['OBJECTID', 'X', 'Y', 'Target'] + FEATURE_COLUMNS
training_csv_path = DIRS['training'] / 'January2024_10Predictors_With_Target.csv'
training_tasks = []

if training_csv_path.exists() and csv_has_columns(training_csv_path, training_selectors, min_rows=1) and not FORCE_RERUN_GEE_EXPORTS:
    log_message("January 2024 training predictor table already exists and is valid.")
else:
    try:
        img2024_train = ten_factor_image(2024, 1, training_region)
        sampled_train = img2024_train.sampleRegions(
            collection=training_points,
            properties=['OBJECTID', 'X', 'Y', 'Target'],
            scale=CELL_SIZE_M,
            projection=projection_30m,
            geometries=False,
            tileScale=16
        )
        task = export_table_to_drive_if_needed(
            sampled_train,
            'January2024_10Predictors_With_Target',
            training_selectors,
            DIRS['training'],
            FORCE_RERUN_GEE_EXPORTS
        )
        if task is not None:
            training_tasks.append(task)
    except Exception as e:
        log_error("GEE_January2024_Training", e)

if WAIT_FOR_GEE_EXPORTS:
    monitor_ee_tasks(training_tasks, poll_seconds=GEE_TASK_POLL_SECONDS)
    move_staged_csvs_to(DIRS['training'])
else:
    print("WAIT_FOR_GEE_EXPORTS is False. Let training export finish, then rerun this cell.")


# Cell 13 — Verify Required GEE CSV Exports

In [ ]:
# ============================================================
# CELL 13 — VERIFY REQUIRED GEE CSV EXPORTS
# ============================================================
# This cell checks if all required CSV files are present before moving to Prophet/XGBoost.

move_staged_csvs_to(DIRS['historical'])
move_staged_csvs_to(DIRS['prophet_input'])
move_staged_csvs_to(DIRS['training'])

missing_hist = []
for year, month in HISTORICAL_YEAR_MONTHS:
    label = month_label(year, month)
    path = DIRS['historical'] / f"{label}.csv"
    if not csv_has_columns(path, hist_selectors, min_rows=1):
        missing_hist.append(path.name)

print("Expected historical CSVs:", EXPECTED_HISTORICAL_FILE_COUNT)
print("Missing/invalid historical CSVs:", len(missing_hist))
if missing_hist:
    print(missing_hist[:20])

print("Static factors valid:", csv_has_columns(static_path, static_selectors, min_rows=1))
print("Training table valid:", csv_has_columns(training_csv_path, training_selectors, min_rows=1))

if missing_hist:
    print("Some historical exports are missing. Rerun Cell 10 after Earth Engine tasks finish.")
else:
    log_message("All historical CSVs are available.")


# Cell 14 — Combine Historical CSVs into Prophet-Ready Table

In [ ]:
# ============================================================
# CELL 14 — COMBINE HISTORICAL CSVs INTO PROPHET-READY TABLE
# ============================================================
combined_path = DIRS['prophet_input'] / 'Combined_50TimeStep_Dynamic_Table.csv'
long_path = DIRS['prophet_input'] / 'Prophet_Ready_Long_Format_Table.csv'

if combined_path.exists() and long_path.exists() and not FORCE_RERUN_PROPHET_BATCHES:
    log_message("Combined Prophet input tables already exist. Loading from Drive.")
    combined_df = safe_read_csv(combined_path)
    long_df = safe_read_csv(long_path)
else:
    dfs = []
    for year, month in HISTORICAL_YEAR_MONTHS:
        label = month_label(year, month)
        path = DIRS['historical'] / f"{label}.csv"
        if not path.exists():
            raise FileNotFoundError(f"Missing historical CSV: {path}")
        dfm = safe_read_csv(path)
        # Ensure standard columns and dates
        dfm['OBJECTID'] = dfm['OBJECTID'].astype(str)
        dfm['Year'] = int(year)
        dfm['Month'] = int(month)
        dfm['Date_Label'] = label
        dfm['ds'] = pd.Timestamp(year=year, month=month, day=1)
        for c in DYNAMIC_FACTORS:
            dfm[c] = pd.to_numeric(dfm[c], errors='coerce')
        dfs.append(dfm[['OBJECTID', 'X', 'Y', 'Year', 'Month', 'Date_Label', 'ds'] + DYNAMIC_FACTORS])

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df = apply_clamps(combined_df, DYNAMIC_FACTORS)
    combined_df = combined_df.sort_values(['OBJECTID', 'ds']).reset_index(drop=True)

    # Interpolate/fill missing dynamic values within each OBJECTID.
    for c in DYNAMIC_FACTORS:
        combined_df[c] = combined_df.groupby('OBJECTID')[c].transform(lambda s: s.interpolate(limit_direction='both'))
        combined_df[c] = combined_df[c].fillna(combined_df[c].median())

    combined_df.to_csv(combined_path, index=False)

    long_parts = []
    for factor in DYNAMIC_FACTORS:
        temp = combined_df[['OBJECTID', 'X', 'Y', 'ds', 'Year', 'Month', 'Date_Label', factor]].copy()
        temp = temp.rename(columns={factor: 'y'})
        temp['Factor'] = factor
        long_parts.append(temp)
    long_df = pd.concat(long_parts, ignore_index=True)
    long_df.to_csv(long_path, index=False)
    log_message("Combined Prophet input tables created.")

print("Combined table shape:", combined_df.shape)
print("Long table shape:", long_df.shape)
print("Number of unique OBJECTIDs:", combined_df['OBJECTID'].nunique())
print("Number of dates:", combined_df['ds'].nunique())
display(combined_df.head())


# Cell 15 — Tune Prophet Parameters by Aggregate Validation

In [ ]:
# ============================================================
# CELL 15 — TUNE PROPHET PARAMETERS BY AGGREGATE VALIDATION
# ============================================================
best_params_path = DIRS['prophet_checkpoints'] / 'prophet_best_params.json'

def fit_prophet_model(train_df, cps, sps, fourier_order):
    m = Prophet(
        growth='linear',
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=cps,
        seasonality_prior_scale=sps,
        uncertainty_samples=0
    )
    m.add_seasonality(name='yearly', period=365.25, fourier_order=int(fourier_order))
    m.fit(train_df[['ds', 'y']])
    return m


def tune_factor_aggregate(factor):
    agg = combined_df.groupby('ds', as_index=False)[factor].mean().rename(columns={factor: 'y'})
    agg = agg.sort_values('ds').reset_index(drop=True)
    train = agg.iloc[:-VALIDATION_STEPS].copy()
    valid = agg.iloc[-VALIDATION_STEPS:].copy()

    records = []
    best = None
    best_rmse = np.inf

    for cps in CHANGEPOINT_PRIOR_OPTIONS:
        for sps in SEASONALITY_PRIOR_OPTIONS:
            for fo in YEARLY_FOURIER_ORDER_OPTIONS:
                try:
                    model = fit_prophet_model(train, cps, sps, fo)
                    pred = model.predict(valid[['ds']])['yhat'].values
                    rmse = math.sqrt(mean_squared_error(valid['y'].values, pred))
                    mae = mean_absolute_error(valid['y'].values, pred)
                    rec = {
                        'Factor': factor,
                        'changepoint_prior_scale': cps,
                        'seasonality_prior_scale': sps,
                        'yearly_fourier_order': fo,
                        'RMSE': rmse,
                        'MAE': mae
                    }
                    records.append(rec)
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best = rec
                except Exception as e:
                    records.append({
                        'Factor': factor,
                        'changepoint_prior_scale': cps,
                        'seasonality_prior_scale': sps,
                        'yearly_fourier_order': fo,
                        'RMSE': np.nan,
                        'MAE': np.nan,
                        'Error': str(e)
                    })
    return best, pd.DataFrame(records)

if best_params_path.exists() and not FORCE_RETUNE_PROPHET:
    best_params = read_json(best_params_path)
    log_message("Loaded saved Prophet best parameters from Drive.")
else:
    best_params = {}
    all_tuning = []
    for factor in DYNAMIC_FACTORS:
        log_message(f"Tuning Prophet aggregate parameters for {factor}...")
        best, tune_df = tune_factor_aggregate(factor)
        all_tuning.append(tune_df)
        if best is None:
            best = {
                'Factor': factor,
                'changepoint_prior_scale': 0.05,
                'seasonality_prior_scale': 10.0,
                'yearly_fourier_order': 3,
                'RMSE': None,
                'MAE': None
            }
        best_params[factor] = {
            'changepoint_prior_scale': float(best['changepoint_prior_scale']),
            'seasonality_prior_scale': float(best['seasonality_prior_scale']),
            'yearly_fourier_order': int(best['yearly_fourier_order'])
        }
        log_message(f"Best Prophet params for {factor}: {best_params[factor]}")

    tuning_results_df = pd.concat(all_tuning, ignore_index=True)
    tuning_results_df.to_csv(DIRS['prophet_checkpoints'] / 'Prophet_Aggregate_Tuning_Results.csv', index=False)
    write_json(best_params_path, best_params)
    log_message("Saved Prophet best parameters.")

print(json.dumps(best_params, indent=2))


# Cell 16 — Prophet Metric Functions and Point-Level Forecasting Functions

In [ ]:
# ============================================================
# CELL 16 — PROPHET METRIC FUNCTIONS AND POINT-LEVEL FORECASTING FUNCTIONS
# ============================================================
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if len(y_true) == 0:
        return {k: np.nan for k in ['R2', 'RMSE', 'MAE', 'Bias', 'MAPE_percent', 'SMAPE_percent', 'WAPE_percent', 'NSE', 'Pearson_r']}

    err = y_pred - y_true
    rmse = math.sqrt(np.mean(err ** 2))
    mae = np.mean(np.abs(err))
    bias = np.mean(err)

    eps = 1e-8
    nz = np.abs(y_true) > eps
    mape = np.mean(np.abs(err[nz] / y_true[nz])) * 100 if nz.sum() > 0 else np.nan

    denom_smape = (np.abs(y_true) + np.abs(y_pred))
    sm_mask = denom_smape > eps
    smape = np.mean(2 * np.abs(err[sm_mask]) / denom_smape[sm_mask]) * 100 if sm_mask.sum() > 0 else np.nan

    wape = np.sum(np.abs(err)) / np.sum(np.abs(y_true)) * 100 if np.sum(np.abs(y_true)) > eps else np.nan

    denom_nse = np.sum((y_true - np.mean(y_true)) ** 2)
    nse = 1 - (np.sum((y_true - y_pred) ** 2) / denom_nse) if denom_nse > eps else np.nan

    r2 = r2_score(y_true, y_pred) if len(y_true) >= 2 and np.nanstd(y_true) > eps else np.nan
    pearson = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) >= 2 and np.std(y_true) > eps and np.std(y_pred) > eps else np.nan

    return {
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Bias': bias,
        'MAPE_percent': mape,
        'SMAPE_percent': smape,
        'WAPE_percent': wape,
        'NSE': nse,
        'Pearson_r': pearson
    }


def fallback_forecast(series_df, target_dates):
    """Fallback when Prophet fails: use simple linear trend over time."""
    df = series_df.sort_values('ds').copy()
    y = pd.to_numeric(df['y'], errors='coerce').interpolate(limit_direction='both').fillna(df['y'].median())
    if y.nunique() <= 1:
        return np.repeat(float(y.mean()), len(target_dates))
    x = (df['ds'] - df['ds'].min()).dt.days.values.astype(float)
    coef = np.polyfit(x, y.values.astype(float), deg=1)
    xt = np.array([(pd.Timestamp(d) - df['ds'].min()).days for d in target_dates], dtype=float)
    return np.polyval(coef, xt)


def point_factor_forecast(point_df, factor, objectid, xcoord, ycoord):
    p = best_params[factor]
    df = point_df[['ds', factor]].rename(columns={factor: 'y'}).copy().sort_values('ds')
    df['y'] = pd.to_numeric(df['y'], errors='coerce').interpolate(limit_direction='both').fillna(df['y'].median())

    valid_dates = df.iloc[-VALIDATION_STEPS:]['ds'].values
    train_df = df.iloc[:-VALIDATION_STEPS].copy()
    valid_df = df.iloc[-VALIDATION_STEPS:].copy()

    # Validation prediction
    validation_pred = None
    method = 'Prophet'
    try:
        if df['y'].nunique() <= 1 or len(train_df) < 10:
            raise ValueError('Constant or too-short series')
        model_v = fit_prophet_model(
            train_df, p['changepoint_prior_scale'], p['seasonality_prior_scale'], p['yearly_fourier_order']
        )
        validation_pred = model_v.predict(valid_df[['ds']])['yhat'].values
    except Exception:
        method = 'Fallback_linear_or_constant'
        validation_pred = fallback_forecast(train_df, valid_df['ds'].tolist())

    metrics = regression_metrics(valid_df['y'].values, validation_pred)
    metrics.update({'OBJECTID': objectid, 'X': xcoord, 'Y': ycoord, 'Factor': factor, 'Method': method})

    validation_records = []
    for ds, obs, pred in zip(valid_df['ds'], valid_df['y'].values, validation_pred):
        validation_records.append({
            'OBJECTID': objectid,
            'X': xcoord,
            'Y': ycoord,
            'Factor': factor,
            'ds': pd.Timestamp(ds),
            'Observed': obs,
            'Predicted': pred,
            'Error': pred - obs,
            'Method': method
        })

    # Final forecast after refitting on all 50 records
    try:
        if df['y'].nunique() <= 1 or len(df) < 10:
            raise ValueError('Constant or too-short series')
        model_f = fit_prophet_model(
            df, p['changepoint_prior_scale'], p['seasonality_prior_scale'], p['yearly_fourier_order']
        )
        future = pd.DataFrame({'ds': FORECAST_TARGET_DATES})
        forecast_values = model_f.predict(future)['yhat'].values
    except Exception:
        forecast_values = fallback_forecast(df, FORECAST_TARGET_DATES)
        method = 'Fallback_linear_or_constant'
        metrics['Method'] = method

    forecast_dict = {'OBJECTID': objectid, 'X': xcoord, 'Y': ycoord}
    for date, val in zip(FORECAST_TARGET_DATES, forecast_values):
        forecast_dict[f'{factor}_{date.year}'] = float(val)

    return forecast_dict, metrics, validation_records


def forecast_one_object(point_df):
    point_df = point_df.sort_values('ds').copy()
    objectid = str(point_df['OBJECTID'].iloc[0])
    xcoord = point_df['X'].iloc[0]
    ycoord = point_df['Y'].iloc[0]

    merged_forecast = {'OBJECTID': objectid, 'X': xcoord, 'Y': ycoord}
    metric_rows = []
    validation_rows = []

    for factor in DYNAMIC_FACTORS:
        try:
            fdict, mdict, vrows = point_factor_forecast(point_df, factor, objectid, xcoord, ycoord)
            for k, v in fdict.items():
                if k not in ['OBJECTID', 'X', 'Y']:
                    merged_forecast[k] = v
            metric_rows.append(mdict)
            validation_rows.extend(vrows)
        except Exception as e:
            log_error(f"Prophet_OBJECTID_{objectid}_{factor}", e)
            for date in FORECAST_TARGET_DATES:
                merged_forecast[f'{factor}_{date.year}'] = np.nan
            metric_rows.append({'OBJECTID': objectid, 'X': xcoord, 'Y': ycoord, 'Factor': factor, 'Method': 'Failed'})

    return merged_forecast, metric_rows, validation_rows

log_message("Prophet forecasting functions are ready.")


# Cell 17 — Resume-Safe Prophet Point-Level Forecasting

In [ ]:
# ============================================================
# CELL 17 — RESUME-SAFE PROPHET POINT-LEVEL FORECASTING
# ============================================================
# This is the heaviest part of the workflow. It fits Prophet per OBJECTID and factor.
# Outputs are saved in batches. If Colab disconnects, rerun this cell and it will
# continue from the missing batches.

batch_dir = DIRS['prophet_checkpoints'] / 'Pointwise_Prophet_Batches'
forecast_batch_dir = batch_dir / 'Forecasts'
metrics_batch_dir = batch_dir / 'Metrics'
validation_batch_dir = batch_dir / 'Validation'
for p in [forecast_batch_dir, metrics_batch_dir, validation_batch_dir]:
    p.mkdir(parents=True, exist_ok=True)

unique_ids = combined_df['OBJECTID'].astype(str).drop_duplicates().tolist()
num_batches = math.ceil(len(unique_ids) / PROPHET_BATCH_SIZE_POINTS)
print("Unique OBJECTIDs:", len(unique_ids))
print("Batch size:", PROPHET_BATCH_SIZE_POINTS)
print("Total batches:", num_batches)

completed_this_run = 0

for b in range(num_batches):
    start_i = b * PROPHET_BATCH_SIZE_POINTS
    end_i = min((b + 1) * PROPHET_BATCH_SIZE_POINTS, len(unique_ids))
    batch_ids = unique_ids[start_i:end_i]
    batch_name = f"batch_{b+1:05d}"

    forecast_file = forecast_batch_dir / f"{batch_name}_forecasts.csv"
    metrics_file = metrics_batch_dir / f"{batch_name}_metrics.csv"
    validation_file = validation_batch_dir / f"{batch_name}_validation.csv"

    required_forecast_cols = ['OBJECTID', 'X', 'Y'] + [f'{fac}_{year}' for fac in DYNAMIC_FACTORS for year in [2030, 2050]]

    if (forecast_file.exists() and metrics_file.exists() and validation_file.exists()
        and csv_has_columns(forecast_file, required_forecast_cols, min_rows=1)
        and not FORCE_RERUN_PROPHET_BATCHES):
        print(f"Skipping completed {batch_name}")
        continue

    log_message(f"Processing Prophet {batch_name}: OBJECTID index {start_i} to {end_i-1}")
    try:
        batch_df = combined_df[combined_df['OBJECTID'].astype(str).isin(batch_ids)].copy()
        grouped = [g for _, g in batch_df.groupby('OBJECTID', sort=False)]

        results = Parallel(n_jobs=N_JOBS_PROPHET, backend='loky')(
            delayed(forecast_one_object)(g) for g in grouped
        )

        forecast_rows = [r[0] for r in results]
        metric_rows = []
        validation_rows = []
        for _, mrows, vrows in results:
            metric_rows.extend(mrows)
            validation_rows.extend(vrows)

        fdf = pd.DataFrame(forecast_rows)
        mdf = pd.DataFrame(metric_rows)
        vdf = pd.DataFrame(validation_rows)

        # Clamp dynamic forecasts to physically reasonable ranges.
        for factor in DYNAMIC_FACTORS:
            for year in [2030, 2050]:
                col = f'{factor}_{year}'
                if col in fdf.columns and factor in CLAMP_RULES:
                    lo, hi = CLAMP_RULES[factor]
                    if lo is not None:
                        fdf[col] = fdf[col].clip(lower=lo)
                    if hi is not None:
                        fdf[col] = fdf[col].clip(upper=hi)

        fdf.to_csv(forecast_file, index=False)
        mdf.to_csv(metrics_file, index=False)
        vdf.to_csv(validation_file, index=False)
        log_message(f"Saved Prophet {batch_name}")
        completed_this_run += 1

        if MAX_PROPHET_BATCHES_PER_RUN is not None and completed_this_run >= MAX_PROPHET_BATCHES_PER_RUN:
            print("Reached MAX_PROPHET_BATCHES_PER_RUN. Rerun this cell later to continue.")
            break

    except Exception as e:
        log_error(f"Prophet_{batch_name}", e)
        print(f"Failed {batch_name}. It will be retried on next run.")

print("Prophet batch processing step finished for this run.")


# Cell 18 — Combine Prophet Forecast Batches and Validation Metrics

In [ ]:
# ============================================================
# CELL 18 — COMBINE PROPHET FORECAST BATCHES AND VALIDATION METRICS
# ============================================================
forecast_2030_path = DIRS['forecast_tables'] / 'SMCI_Prophet_January_2030_30m_Point_Table.csv'
forecast_2050_path = DIRS['forecast_tables'] / 'SMCI_Prophet_January_2050_30m_Point_Table.csv'
point_metrics_path = DIRS['prophet_metrics'] / 'Prophet_Point_Level_Metrics.csv'
factor_metrics_path = DIRS['prophet_metrics'] / 'Prophet_Factor_Summary_Metrics.csv'
overall_metrics_path = DIRS['prophet_metrics'] / 'Prophet_Overall_Metrics.csv'
validation_obs_pred_path = DIRS['prophet_metrics'] / 'Prophet_Validation_Observed_vs_Predicted.csv'

forecast_files = sorted((DIRS['prophet_checkpoints'] / 'Pointwise_Prophet_Batches' / 'Forecasts').glob('*_forecasts.csv'))
metrics_files = sorted((DIRS['prophet_checkpoints'] / 'Pointwise_Prophet_Batches' / 'Metrics').glob('*_metrics.csv'))
validation_files = sorted((DIRS['prophet_checkpoints'] / 'Pointwise_Prophet_Batches' / 'Validation').glob('*_validation.csv'))

print("Forecast batches found:", len(forecast_files))
print("Metrics batches found:", len(metrics_files))
print("Validation batches found:", len(validation_files))

if len(forecast_files) == 0:
    raise FileNotFoundError("No Prophet forecast batches found. Complete Cell 17 first.")

forecast_all = pd.concat([safe_read_csv(p) for p in forecast_files], ignore_index=True)
forecast_all['OBJECTID'] = forecast_all['OBJECTID'].astype(str)

# Load static factors
static_df = safe_read_csv(static_path)
static_df['OBJECTID'] = static_df['OBJECTID'].astype(str)
for c in STATIC_FACTORS:
    static_df[c] = pd.to_numeric(static_df[c], errors='coerce')
static_df, static_medians = clean_numeric_features(static_df, STATIC_FACTORS)

# Create 2030 and 2050 predictor tables
for target_year, out_path in [(2030, forecast_2030_path), (2050, forecast_2050_path)]:
    cols = ['OBJECTID', 'X', 'Y'] + [f'{fac}_{target_year}' for fac in DYNAMIC_FACTORS]
    temp = forecast_all[cols].copy()
    rename_map = {f'{fac}_{target_year}': fac for fac in DYNAMIC_FACTORS}
    temp = temp.rename(columns=rename_map)
    temp = apply_clamps(temp, DYNAMIC_FACTORS)
    temp = temp.merge(static_df[['OBJECTID'] + STATIC_FACTORS], on='OBJECTID', how='left')
    temp, _ = clean_numeric_features(temp, FEATURE_COLUMNS)
    temp.to_csv(out_path, index=False)
    print("Saved:", out_path, temp.shape)

# Metrics tables
if metrics_files:
    metrics_all = pd.concat([safe_read_csv(p) for p in metrics_files], ignore_index=True)
    metrics_all.to_csv(point_metrics_path, index=False)

    metric_cols = ['R2', 'RMSE', 'MAE', 'Bias', 'MAPE_percent', 'SMAPE_percent', 'WAPE_percent', 'NSE', 'Pearson_r']
    summary = metrics_all.groupby('Factor')[metric_cols].agg(['mean', 'median', 'std', 'min', 'max'])
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary = summary.reset_index()
    summary.to_csv(factor_metrics_path, index=False)

    overall = pd.DataFrame({
        'Metric': metric_cols,
        'Mean_Value': [metrics_all[c].mean() for c in metric_cols],
        'Median_Value': [metrics_all[c].median() for c in metric_cols]
    })
    overall.to_csv(overall_metrics_path, index=False)
    display(summary)

# Validation observed vs predicted can be large; combine only if not already combined or forced.
if validation_files and (not validation_obs_pred_path.exists() or FORCE_RERUN_PROPHET_BATCHES):
    with open(validation_obs_pred_path, 'w', encoding='utf-8') as fout:
        first = True
        for p in validation_files:
            chunk = safe_read_csv(p)
            chunk.to_csv(fout, index=False, header=first)
            first = False
    print("Saved combined validation observed-vs-predicted:", validation_obs_pred_path)

print("Prophet forecast and metric combination completed.")


# Cell 19 — Prepare January 2024 Full-Grid Predictor Table

In [ ]:
# ============================================================
# CELL 19 — PREPARE JANUARY 2024 FULL-GRID PREDICTOR TABLE
# ============================================================
# This creates the full 30 m grid predictor table for January 2024 using the
# historical January 2024 dynamic factors plus static factors.

jan2024_full_path = DIRS['forecast_tables'] / 'SMCI_January_2024_30m_Point_Table.csv'

if jan2024_full_path.exists() and csv_has_columns(jan2024_full_path, ['OBJECTID', 'X', 'Y'] + FEATURE_COLUMNS, min_rows=1) and not FORCE_RERUN_PROPHET_BATCHES:
    jan2024_full_df = safe_read_csv(jan2024_full_path)
    log_message("Loaded existing January 2024 full-grid predictor table.")
else:
    jan2024_dynamic_path = DIRS['historical'] / 'JANUARY_2024.csv'
    jan2024_dynamic = safe_read_csv(jan2024_dynamic_path)
    jan2024_dynamic['OBJECTID'] = jan2024_dynamic['OBJECTID'].astype(str)
    jan2024_dynamic = apply_clamps(jan2024_dynamic, DYNAMIC_FACTORS)

    static_df = safe_read_csv(static_path)
    static_df['OBJECTID'] = static_df['OBJECTID'].astype(str)
    static_df, _ = clean_numeric_features(static_df, STATIC_FACTORS)

    jan2024_full_df = jan2024_dynamic[['OBJECTID', 'X', 'Y'] + DYNAMIC_FACTORS].merge(
        static_df[['OBJECTID'] + STATIC_FACTORS], on='OBJECTID', how='left'
    )
    jan2024_full_df, _ = clean_numeric_features(jan2024_full_df, FEATURE_COLUMNS)
    jan2024_full_df.to_csv(jan2024_full_path, index=False)
    log_message("Saved January 2024 full-grid predictor table.")

print("January 2024 full-grid table:", jan2024_full_df.shape)
display(jan2024_full_df.head())


# Cell 20 — Load and Clean January 2024 Training Dataset with Target or BC

In [ ]:
# ============================================================
# CELL 20 — LOAD AND CLEAN JANUARY 2024 TRAINING DATASET WITH TARGET OR BC
# ============================================================
training_df = safe_read_csv(training_csv_path)
training_df['OBJECTID'] = training_df['OBJECTID'].astype(str)

# Detect Target or BC. The GEE export standardises BC to Target, but this remains flexible.
target_col = None
for c in TARGET_CANDIDATES:
    if c in training_df.columns:
        target_col = c
        break
if target_col is None:
    raise ValueError("No target column found. Expected either 'Target' or 'BC'.")

training_df['Target'] = pd.to_numeric(training_df[target_col], errors='coerce')
training_df = training_df.dropna(subset=['Target']).copy()
training_df['Target'] = training_df['Target'].astype(int)

found_classes = sorted(training_df['Target'].unique().tolist())
print("Target column used:", target_col)
print("Detected classes:", found_classes)
print("Class distribution:")
print(training_df['Target'].value_counts().sort_index())

expected_classes = [0, 1, 2]
if found_classes != expected_classes:
    raise ValueError(f"Target must contain exactly classes {expected_classes}. Detected: {found_classes}")

# Enforce exact 10 predictors
missing_features = [c for c in FEATURE_COLUMNS if c not in training_df.columns]
if missing_features:
    raise ValueError(f"Missing predictors in training data: {missing_features}")

training_df = apply_clamps(training_df, DYNAMIC_FACTORS)
training_df, feature_medians = clean_numeric_features(training_df, FEATURE_COLUMNS)

clean_training_path = DIRS['training'] / 'January2024_10Predictors_Cleaned.csv'
training_df[['OBJECTID', 'X', 'Y', 'Target'] + FEATURE_COLUMNS].to_csv(clean_training_path, index=False)

print("Clean training data shape:", training_df.shape)
display(training_df[['OBJECTID', 'X', 'Y', 'Target'] + FEATURE_COLUMNS].head())


# Cell 21 — Pearson Correlation Matrix of the 10 Predictors

In [ ]:
# ============================================================
# CELL 21 — PEARSON CORRELATION MATRIX OF THE 10 PREDICTORS
# ============================================================
pearson_csv = DIRS['figures'] / 'Pearson_Correlation_Matrix_of_10_SMCI_Predictors.csv'
pearson_png = DIRS['figures'] / 'Pearson_Correlation_Matrix_of_10_SMCI_Predictors.png'

X_corr = training_df[FEATURE_COLUMNS].copy()
pearson_corr = X_corr.corr(method='pearson')
pearson_corr.to_csv(pearson_csv)

if not pearson_png.exists() or FORCE_REMAKE_FIGURES:
    fig, ax = plt.subplots(figsize=(12, 10))
    mask_upper = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)
    masked_corr = np.ma.masked_where(mask_upper, pearson_corr.values)

    cmap_corr = plt.cm.coolwarm.copy()
    cmap_corr.set_bad(color='white')

    im = ax.imshow(masked_corr, cmap=cmap_corr, vmin=-1, vmax=1)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Pearson correlation coefficient', fontsize=14, fontweight='bold')
    cbar.ax.tick_params(labelsize=12)

    ax.set_xticks(np.arange(len(FEATURE_COLUMNS)))
    ax.set_yticks(np.arange(len(FEATURE_COLUMNS)))
    ax.set_xticklabels(FEATURE_COLUMNS, rotation=45, ha='right', fontsize=13, fontweight='bold')
    ax.set_yticklabels(FEATURE_COLUMNS, fontsize=13, fontweight='bold')

    for i in range(len(FEATURE_COLUMNS)):
        for j in range(len(FEATURE_COLUMNS)):
            if i > j:
                value = pearson_corr.iloc[i, j]
                text_color = 'white' if abs(value) >= 0.65 else 'black'
                ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=12, fontweight='bold', color=text_color)

    ax.set_title('Pearson Correlation Matrix of Input Variables', fontsize=20, fontweight='bold', pad=16)
    fig.tight_layout()
    fig.savefig(pearson_png, dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Pearson figure already exists:", pearson_png)

print("Saved Pearson outputs.")


# Cell 22 — Train/Test Split Shared by XGBoost and MLP-ANN

In [ ]:
# ============================================================
# CELL 22 — TRAIN/TEST SPLIT SHARED BY XGBOOST AND MLP-ANN
# ============================================================
X = training_df[FEATURE_COLUMNS].copy()
y = training_df['Target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

classes = np.array([0, 1, 2])
y_train_bin = label_binarize(y_train, classes=classes)
y_test_bin = label_binarize(y_test, classes=classes)

split_report = pd.DataFrame({
    'Dataset': ['Training', 'Testing'],
    'Records': [len(y_train), len(y_test)],
    'Percentage': [100 * len(y_train) / len(y), 100 * len(y_test) / len(y)]
})
split_report.to_csv(DIRS['training'] / 'Train_Test_Split_Report.csv', index=False)

display(split_report)
print("Training class distribution:")
print(y_train.value_counts().sort_index())
print("Testing class distribution:")
print(y_test.value_counts().sort_index())


# Cell 23 — Train and Validate XGBoost Classifier

In [ ]:
# ============================================================
# CELL 23 — TRAIN AND VALIDATE XGBOOST CLASSIFIER
# ============================================================
xgb_model_path = DIRS['xgb'] / 'XGBoost_SMCI_Model.joblib'
xgb_params_path = DIRS['xgb'] / 'XGBoost_Best_Parameters.json'
xgb_perf_path = DIRS['xgb'] / 'XGBoost_SMCI_Model_Performance_Metrics.csv'

base_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': 1
}

if xgb_model_path.exists() and xgb_params_path.exists() and not FORCE_RETRAIN_XGBOOST:
    best_xgb = load(xgb_model_path)
    best_params_xgb = read_json(xgb_params_path)
    log_message("Loaded existing XGBoost model from Drive.")
else:
    xgb_base = XGBClassifier(**base_params)
    param_distributions = {
        'n_estimators': randint(150, 500),
        'max_depth': randint(3, 8),
        'learning_rate': uniform(0.01, 0.12),
        'subsample': uniform(0.70, 0.30),
        'colsample_bytree': uniform(0.70, 0.30),
        'gamma': uniform(0.0, 0.40),
        'reg_alpha': uniform(0.0, 0.30),
        'reg_lambda': uniform(1.0, 3.0),
        'min_child_weight': randint(1, 8)
    }
    cv_search = StratifiedKFold(n_splits=XGB_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    random_search = RandomizedSearchCV(
        estimator=xgb_base,
        param_distributions=param_distributions,
        n_iter=XGB_RANDOM_SEARCH_ITERATIONS,
        scoring='roc_auc_ovr',
        cv=cv_search,
        verbose=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )
    log_message("Starting XGBoost hyperparameter tuning...")
    random_search.fit(X_train, y_train)
    best_params_xgb = random_search.best_params_

    best_xgb = XGBClassifier(**base_params, **best_params_xgb)
    best_xgb.set_params(n_jobs=-1)
    best_xgb.fit(X_train, y_train)

    dump(best_xgb, xgb_model_path)
    write_json(xgb_params_path, best_params_xgb)
    pd.DataFrame(random_search.cv_results_).to_csv(DIRS['xgb'] / 'XGBoost_SMCI_Random_Search_CV_Results.csv', index=False)
    log_message("XGBoost training completed and saved.")

# Predictions and performance
y_train_pred_xgb = best_xgb.predict(X_train)
y_test_pred_xgb = best_xgb.predict(X_test)
y_train_prob_xgb = best_xgb.predict_proba(X_train)
y_test_prob_xgb = best_xgb.predict_proba(X_test)


def multiclass_performance(y_true, y_pred, y_prob, y_bin, dataset_name, model_name):
    return {
        'Model': model_name,
        'Dataset': dataset_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Precision_Macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Precision_Weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall_Macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall_Weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1_Macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'F1_Weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'ROC_AUC_OVR_Macro': roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'),
        'ROC_AUC_OVR_Weighted': roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted'),
        'ROC_AUC_Micro': roc_auc_score(y_bin.ravel(), y_prob.ravel()),
        'Log_Loss': log_loss(y_true, y_prob)
    }

xgb_performance_df = pd.DataFrame([
    multiclass_performance(y_train, y_train_pred_xgb, y_train_prob_xgb, y_train_bin, 'Training', 'XGBoost'),
    multiclass_performance(y_test, y_test_pred_xgb, y_test_prob_xgb, y_test_bin, 'Testing', 'XGBoost')
])
xgb_performance_df.to_csv(xgb_perf_path, index=False)

testing_report_xgb = pd.DataFrame(classification_report(y_test, y_test_pred_xgb, output_dict=True, zero_division=0)).transpose()
testing_report_xgb.to_csv(DIRS['xgb'] / 'XGBoost_SMCI_Testing_Classification_Report.csv')

display(xgb_performance_df)
display(testing_report_xgb)


# Cell 24 — XGBoost 5-Fold Verification Plot

In [ ]:
# ============================================================
# CELL 24 — XGBOOST 5-FOLD VERIFICATION PLOT
# ============================================================
cv_diag_path = DIRS['xgb'] / 'XGBoost_SMCI_5Fold_CV_Diagnostic.csv'
cv_summary_path = DIRS['xgb'] / 'XGBoost_SMCI_5Fold_CV_Summary.csv'
cv_plot_path = DIRS['figures'] / 'Five_Fold_Verification_Plot_XGBoost.png'

cv_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring_metrics = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro', 'roc_auc_ovr': 'roc_auc_ovr'}

cv_scores = cross_validate(best_xgb, X, y, cv=cv_eval, scoring=scoring_metrics, n_jobs=-1, return_train_score=False)
cv_diagnostic_df = pd.DataFrame({
    'Fold': np.arange(1, 6),
    'Accuracy': cv_scores['test_accuracy'],
    'F1_Macro': cv_scores['test_f1_macro'],
    'ROC_AUC_OVR': cv_scores['test_roc_auc_ovr']
})
cv_summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1_Macro', 'ROC_AUC_OVR'],
    'Mean': [cv_diagnostic_df['Accuracy'].mean(), cv_diagnostic_df['F1_Macro'].mean(), cv_diagnostic_df['ROC_AUC_OVR'].mean()],
    'Standard_Deviation': [cv_diagnostic_df['Accuracy'].std(), cv_diagnostic_df['F1_Macro'].std(), cv_diagnostic_df['ROC_AUC_OVR'].std()]
})
cv_diagnostic_df.to_csv(cv_diag_path, index=False)
cv_summary_df.to_csv(cv_summary_path, index=False)

if not cv_plot_path.exists() or FORCE_REMAKE_FIGURES:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.plot(cv_diagnostic_df['Fold'], cv_diagnostic_df['Accuracy'], marker='o', linewidth=2.8, markersize=8, label='Accuracy')
    ax.plot(cv_diagnostic_df['Fold'], cv_diagnostic_df['F1_Macro'], marker='s', linewidth=2.8, markersize=8, label='F1 macro')
    ax.plot(cv_diagnostic_df['Fold'], cv_diagnostic_df['ROC_AUC_OVR'], marker='^', linewidth=2.8, markersize=8, label='ROC-AUC OVR')
    ax.set_xticks(cv_diagnostic_df['Fold'])
    ax.set_xlabel('Fold number', fontweight='bold')
    ax.set_ylabel('Verification metric value', fontweight='bold')
    ax.set_title('Five-Fold Cross-Validation Verification for XGBoost', fontweight='bold')
    ax.grid(True, alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(cv_plot_path, dpi=300, bbox_inches='tight')
    plt.show()

display(cv_diagnostic_df)
display(cv_summary_df)


# Cell 25 — XGBoost Confusion Matrix, ROC Curve and Feature Importance

In [ ]:
# ============================================================
# CELL 25 — XGBOOST CONFUSION MATRIX, ROC CURVE AND FEATURE IMPORTANCE
# ============================================================
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred_xgb, labels=classes)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"Actual_{i}_{CLASS_LABELS[i]}" for i in classes],
    columns=[f"Predicted_{i}_{CLASS_LABELS[i]}" for i in classes]
)
confusion_matrix_df.to_csv(DIRS['xgb'] / 'XGBoost_SMCI_Confusion_Matrix.csv')

cm_png = DIRS['figures'] / 'XGBoost_Testing_Confusion_Matrix.png'
if not cm_png.exists() or FORCE_REMAKE_FIGURES:
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=12)
    ax.set_title('XGBoost Testing Confusion Matrix', fontsize=20, fontweight='bold')
    ax.set_xticks(classes)
    ax.set_yticks(classes)
    ax.set_xticklabels([str(i) for i in classes], fontsize=13, fontweight='bold')
    ax.set_yticklabels([str(i) for i in classes], fontsize=13, fontweight='bold')
    threshold = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                    color='white' if cm[i, j] > threshold else 'black', fontsize=16, fontweight='bold')
    ax.set_xlabel('Predicted label', fontweight='bold')
    ax.set_ylabel('True label', fontweight='bold')
    fig.tight_layout()
    fig.savefig(cm_png, dpi=300, bbox_inches='tight')
    plt.show()

# Micro-average ROC curve
fpr_micro, tpr_micro, thresholds_micro = roc_curve(y_test_bin.ravel(), y_test_prob_xgb.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)
roc_curve_df = pd.DataFrame({'False_Positive_Rate_Micro': fpr_micro, 'True_Positive_Rate_Micro': tpr_micro, 'Threshold_Micro': thresholds_micro})
roc_curve_df.to_csv(DIRS['xgb'] / 'XGBoost_SMCI_ROC_Curve_Values.csv', index=False)

roc_png = DIRS['figures'] / 'XGBoost_ROC_AUC_Curve.png'
if not roc_png.exists() or FORCE_REMAKE_FIGURES:
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.plot(fpr_micro, tpr_micro, linewidth=2.8, label=f'Micro-average AUC = {roc_auc_micro:.3f}')
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1.5)
    ax.set_xlabel('False positive rate', fontweight='bold')
    ax.set_ylabel('True positive rate', fontweight='bold')
    ax.set_title('ROC-AUC Curve for XGBoost Classification', fontsize=20, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.35)
    fig.tight_layout()
    fig.savefig(roc_png, dpi=300, bbox_inches='tight')
    plt.show()

# Feature importance
booster = best_xgb.get_booster()
importance_gain_dict = booster.get_score(importance_type='gain')
importance_weight_dict = booster.get_score(importance_type='weight')
importance_cover_dict = booster.get_score(importance_type='cover')
feature_importance_df = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Gain': [importance_gain_dict.get(f, 0) for f in FEATURE_COLUMNS],
    'Weight': [importance_weight_dict.get(f, 0) for f in FEATURE_COLUMNS],
    'Cover': [importance_cover_dict.get(f, 0) for f in FEATURE_COLUMNS]
})
feature_importance_df['Gain_Normalized_Percent'] = feature_importance_df['Gain'] / feature_importance_df['Gain'].sum() * 100
feature_importance_df = feature_importance_df.sort_values('Gain', ascending=False)
feature_importance_df.to_csv(DIRS['xgb'] / 'XGBoost_SMCI_Feature_Importance.csv', index=False)

fi_png = DIRS['figures'] / 'XGBoost_Feature_Importance.png'
if not fi_png.exists() or FORCE_REMAKE_FIGURES:
    plot_df = feature_importance_df.sort_values('Gain', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(plot_df['Feature'], plot_df['Gain'])
    max_gain = plot_df['Gain'].max() if plot_df['Gain'].max() > 0 else 1
    for bar in bars:
        width = bar.get_width()
        ax.text(width + max_gain * 0.01, bar.get_y() + bar.get_height()/2,
                f'{width:.4f}', va='center', fontsize=11, fontweight='bold')
    ax.set_xlabel('Gain-based feature importance', fontweight='bold')
    ax.set_title('XGBoost Global Feature Importance', fontsize=20, fontweight='bold')
    fig.tight_layout()
    fig.savefig(fi_png, dpi=300, bbox_inches='tight')
    plt.show()

display(feature_importance_df)


# Cell 26 — XGBoost SHAP Beeswarm, Summary Bar, Dependence, Heatmap and Interactions

In [ ]:
# ============================================================
# CELL 26 — XGBOOST SHAP BEESWARM, SUMMARY BAR, DEPENDENCE, HEATMAP AND INTERACTIONS
# ============================================================
# Uses a sampled test set for speed and stability.

shap_sample_n = min(SHAP_SAMPLE_SIZE, X_test.shape[0])
X_shap = X_test.sample(n=shap_sample_n, random_state=RANDOM_STATE)
y_shap_pred = best_xgb.predict(X_shap)

explainer = shap.TreeExplainer(best_xgb)
raw_shap_values = explainer.shap_values(X_shap)
expected_values_raw = explainer.expected_value


def standardize_multiclass_shap(raw_values, n_classes, n_samples, n_features):
    if isinstance(raw_values, list):
        return np.array(raw_values)
    arr = np.array(raw_values)
    if arr.ndim == 3:
        if arr.shape[0] == n_classes:
            return arr
        if arr.shape[2] == n_classes:
            return np.transpose(arr, (2, 0, 1))
        if arr.shape[1] == n_classes:
            return np.transpose(arr, (1, 0, 2))
    if arr.ndim == 2:
        return arr.reshape(1, n_samples, n_features)
    raise ValueError(f"Unexpected SHAP value shape: {arr.shape}")

shap_values_by_class = standardize_multiclass_shap(raw_shap_values, 3, X_shap.shape[0], X_shap.shape[1])
expected_values = np.array(expected_values_raw)
if expected_values.ndim == 0:
    expected_values = np.repeat(float(expected_values), 3)

# Composite SHAP based on predicted class for each sample.
composite_shap_values = np.vstack([
    shap_values_by_class[int(y_shap_pred[i]), i, :]
    for i in range(len(y_shap_pred))
])

shap_importance_df = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Mean_ABS_SHAP_All_Classes': np.abs(shap_values_by_class).mean(axis=(0, 1)),
    'Mean_ABS_SHAP_Predicted_Class': np.abs(composite_shap_values).mean(axis=0)
}).sort_values('Mean_ABS_SHAP_All_Classes', ascending=False)
shap_importance_df.to_csv(DIRS['xgb'] / 'XGBoost_SMCI_SHAP_Feature_Importance.csv', index=False)

# Beeswarm
beeswarm_png = DIRS['figures'] / 'SHAP_Beeswarm_XGBoost.png'
if not beeswarm_png.exists() or FORCE_REMAKE_FIGURES:
    plt.figure(figsize=(10, 8))
    shap.summary_plot(composite_shap_values, X_shap, feature_names=FEATURE_COLUMNS, show=False, max_display=len(FEATURE_COLUMNS))
    plt.title('SHAP Beeswarm Plot for XGBoost Classification', fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.savefig(beeswarm_png, dpi=300, bbox_inches='tight')
    plt.show()

# Summary bar plot
bar_png = DIRS['figures'] / 'SHAP_Summary_Bar_XGBoost.png'
if not bar_png.exists() or FORCE_REMAKE_FIGURES:
    plt.figure(figsize=(10, 7))
    shap.summary_plot(composite_shap_values, X_shap, feature_names=FEATURE_COLUMNS, plot_type='bar', show=False, max_display=len(FEATURE_COLUMNS))
    plt.title('SHAP Summary Bar Plot for XGBoost Classification', fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.savefig(bar_png, dpi=300, bbox_inches='tight')
    plt.show()

# Dependence plots for all predictors
for feature in FEATURE_COLUMNS:
    dep_png = FIG_SUBDIRS['shap_dependence'] / f'SHAP_Dependence_{feature}_XGBoost.png'
    if dep_png.exists() and not FORCE_REMAKE_FIGURES:
        continue
    try:
        plt.figure(figsize=(8, 6))
        shap.dependence_plot(feature, composite_shap_values, X_shap, feature_names=FEATURE_COLUMNS, show=False)
        plt.title(f'SHAP Dependence Plot: {feature}', fontsize=17, fontweight='bold')
        plt.tight_layout()
        plt.savefig(dep_png, dpi=300, bbox_inches='tight')
        plt.show()
    except Exception as e:
        log_error(f"SHAP_Dependence_{feature}", e)

# SHAP heatmap using an Explanation object
heatmap_png = DIRS['figures'] / 'SHAP_Heatmap_XGBoost.png'
if not heatmap_png.exists() or FORCE_REMAKE_FIGURES:
    try:
        explanation = shap.Explanation(
            values=composite_shap_values,
            base_values=np.zeros(composite_shap_values.shape[0]),
            data=X_shap.values,
            feature_names=FEATURE_COLUMNS
        )
        shap.plots.heatmap(explanation, max_display=len(FEATURE_COLUMNS), show=False)
        plt.title('SHAP Heatmap for XGBoost Classification', fontsize=18, fontweight='bold')
        plt.tight_layout()
        plt.savefig(heatmap_png, dpi=300, bbox_inches='tight')
        plt.show()
    except Exception as e:
        log_error("SHAP_Heatmap", e)
        # Fallback heatmap
        ordered = np.argsort(np.abs(composite_shap_values).mean(axis=1))[::-1]
        fig, ax = plt.subplots(figsize=(12, 8))
        im = ax.imshow(composite_shap_values[ordered, :], aspect='auto', cmap='coolwarm')
        ax.set_xticks(np.arange(len(FEATURE_COLUMNS)))
        ax.set_xticklabels(FEATURE_COLUMNS, rotation=45, ha='right', fontweight='bold')
        ax.set_ylabel('Sample order', fontweight='bold')
        ax.set_title('SHAP Heatmap for XGBoost Classification', fontsize=18, fontweight='bold')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='SHAP value')
        fig.tight_layout()
        fig.savefig(heatmap_png, dpi=300, bbox_inches='tight')
        plt.show()

# SHAP interaction heatmap
interaction_png = DIRS['figures'] / 'SHAP_Interaction_Heatmap_XGBoost.png'
interaction_csv = DIRS['xgb'] / 'XGBoost_SMCI_SHAP_Interaction_Matrix.csv'
if not interaction_png.exists() or FORCE_REMAKE_FIGURES:
    interaction_sample_n = min(SHAP_INTERACTION_SAMPLE_SIZE, X_shap.shape[0])
    X_interaction = X_shap.sample(n=interaction_sample_n, random_state=RANDOM_STATE)
    interaction_method = 'True SHAP interaction values'
    try:
        raw_interaction_values = explainer.shap_interaction_values(X_interaction)
        arr = np.array(raw_interaction_values)
        if isinstance(raw_interaction_values, list):
            interaction_values_by_class = np.array(raw_interaction_values)
        elif arr.ndim == 4 and arr.shape[3] == 3:
            interaction_values_by_class = np.transpose(arr, (3, 0, 1, 2))
        elif arr.ndim == 4 and arr.shape[0] == 3:
            interaction_values_by_class = arr
        else:
            raise ValueError(f"Unexpected interaction shape: {arr.shape}")
        interaction_matrix = np.abs(interaction_values_by_class).mean(axis=(0, 1))
    except Exception as e:
        interaction_method = 'Fallback SHAP-weighted correlation approximation'
        log_error("SHAP_Interaction_Fallback", e)
        shap_abs_df = pd.DataFrame(np.abs(composite_shap_values), columns=FEATURE_COLUMNS)
        interaction_matrix = np.zeros((len(FEATURE_COLUMNS), len(FEATURE_COLUMNS)))
        for i, fi in enumerate(FEATURE_COLUMNS):
            for j, fj in enumerate(FEATURE_COLUMNS):
                if i == j:
                    interaction_matrix[i, j] = shap_abs_df[fi].mean()
                else:
                    corr_value = np.corrcoef(shap_abs_df[fi], shap_abs_df[fj])[0, 1]
                    if np.isnan(corr_value):
                        corr_value = 0
                    interaction_matrix[i, j] = abs(corr_value) * (shap_abs_df[fi].mean() + shap_abs_df[fj].mean()) / 2

    interaction_matrix_df = pd.DataFrame(interaction_matrix, index=FEATURE_COLUMNS, columns=FEATURE_COLUMNS)
    interaction_matrix_df.to_csv(interaction_csv)
    with open(DIRS['xgb'] / 'SHAP_Interaction_Method.txt', 'w') as f:
        f.write(interaction_method)

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(interaction_matrix_df.values, cmap='Greys', aspect='auto')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Mean absolute SHAP interaction value', fontsize=13, fontweight='bold')
    ax.set_xticks(np.arange(len(FEATURE_COLUMNS)))
    ax.set_yticks(np.arange(len(FEATURE_COLUMNS)))
    ax.set_xticklabels(FEATURE_COLUMNS, rotation=45, ha='right', fontsize=12, fontweight='bold')
    ax.set_yticklabels(FEATURE_COLUMNS, fontsize=12, fontweight='bold')
    threshold = interaction_matrix_df.values.max() * 0.55 if interaction_matrix_df.values.max() > 0 else 0
    for i in range(len(FEATURE_COLUMNS)):
        for j in range(len(FEATURE_COLUMNS)):
            val = interaction_matrix_df.iloc[i, j]
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white' if val > threshold else 'black')
    ax.set_title('SHAP Feature Interaction Heatmap', fontsize=20, fontweight='bold')
    fig.tight_layout()
    fig.savefig(interaction_png, dpi=300, bbox_inches='tight')
    plt.show()

display(shap_importance_df)


# Cell 27 — XGBoost SHAP Waterfall Plots for Selected Samples

In [ ]:
# ============================================================
# CELL 27 — XGBOOST SHAP WATERFALL PLOTS FOR SELECTED SAMPLES
# ============================================================
# One representative sample is selected for each predicted class where possible.

for class_id in classes:
    try:
        idx_candidates = np.where(y_shap_pred == class_id)[0]
        if len(idx_candidates) == 0:
            print(f"No SHAP sample predicted as class {class_id}; skipping.")
            continue
        sample_index = idx_candidates[0]
        waterfall_png = FIG_SUBDIRS['waterfall'] / f'SHAP_Waterfall_Class_{class_id}_{CLASS_LABELS[class_id].replace(" ", "_")}.png'
        if waterfall_png.exists() and not FORCE_REMAKE_FIGURES:
            continue

        explanation = shap.Explanation(
            values=shap_values_by_class[class_id, sample_index, :],
            base_values=expected_values[class_id] if len(expected_values) > class_id else 0,
            data=X_shap.iloc[sample_index].values,
            feature_names=FEATURE_COLUMNS
        )
        plt.figure(figsize=(9, 7))
        shap.plots.waterfall(explanation, max_display=len(FEATURE_COLUMNS), show=False)
        plt.title(f'SHAP Waterfall Plot: {CLASS_LABELS[class_id]}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(waterfall_png, dpi=300, bbox_inches='tight')
        plt.show()
    except Exception as e:
        log_error(f"SHAP_Waterfall_Class_{class_id}", e)


# Cell 28 — Optional SHAP-IQ Advanced Interaction Analysis

In [ ]:
# ============================================================
# CELL 28 — OPTIONAL SHAP-IQ ADVANCED INTERACTION ANALYSIS
# ============================================================
# SHAP-IQ is optional and can be computationally heavy. This cell tries to run it
# safely. If it fails due to package/API/runtime issues, the rest of the workflow continues.

shapiq_status_path = FIG_SUBDIRS['shapiq'] / 'SHAP_IQ_Status.txt'
shapiq_csv_path = FIG_SUBDIRS['shapiq'] / 'SHAP_IQ_Interaction_Summary.csv'

try:
    import shapiq
    status_lines = []
    status_lines.append(f"SHAP-IQ package imported successfully at {now_string()}.")

    X_iq_bg = X_train.sample(n=min(200, X_train.shape[0]), random_state=RANDOM_STATE)
    X_iq_explain = X_test.sample(n=min(SHAP_IQ_SAMPLE_SIZE, X_test.shape[0]), random_state=RANDOM_STATE)

    # Focus on probability of class 2 (Moist zones) because it is the highest moisture condition.
    def predict_moist_probability(arr):
        arr_df = pd.DataFrame(arr, columns=FEATURE_COLUMNS)
        return best_xgb.predict_proba(arr_df)[:, 2]

    # The SHAP-IQ API may change across versions. Try a common interface, otherwise log and skip.
    try:
        explainer_iq = shapiq.Explainer(
            model=predict_moist_probability,
            data=X_iq_bg.values,
            index='k-SII',
            max_order=2
        )
        interaction_records = []
        for row_i in range(X_iq_explain.shape[0]):
            try:
                iv = explainer_iq.explain(X_iq_explain.iloc[row_i].values, budget=2**len(FEATURE_COLUMNS))
                # Try converting top interactions to a dataframe if supported.
                if hasattr(iv, 'dict_values'):
                    vals = iv.dict_values
                    for key, value in vals.items():
                        interaction_records.append({'Sample_Index': row_i, 'Interaction_Key': str(key), 'Value': value})
                elif hasattr(iv, 'values'):
                    interaction_records.append({'Sample_Index': row_i, 'Interaction_Key': 'values_array', 'Value': np.nan})
                else:
                    interaction_records.append({'Sample_Index': row_i, 'Interaction_Key': 'unknown_output', 'Value': np.nan})
            except Exception as row_e:
                interaction_records.append({'Sample_Index': row_i, 'Interaction_Key': 'row_failed', 'Value': np.nan, 'Error': str(row_e)})

        shapiq_df = pd.DataFrame(interaction_records)
        shapiq_df.to_csv(shapiq_csv_path, index=False)
        status_lines.append("SHAP-IQ execution completed. See SHAP_IQ_Interaction_Summary.csv.")
    except Exception as api_e:
        status_lines.append("SHAP-IQ could not be executed with the installed package API.")
        status_lines.append(str(api_e))
        log_error("SHAP_IQ_API", api_e)

    with open(shapiq_status_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(status_lines))
    print('\n'.join(status_lines))

except Exception as e:
    with open(shapiq_status_path, 'w', encoding='utf-8') as f:
        f.write(f"SHAP-IQ skipped or failed at {now_string()}.\n{str(e)}")
    log_error("SHAP_IQ_Import", e)
    print("SHAP-IQ skipped. The rest of the workflow continues.")


# Cell 29 — Train and Validate MLP-ANN Deep Learning Classifier

In [ ]:
# ============================================================
# CELL 29 — TRAIN AND VALIDATE MLP-ANN DEEP LEARNING CLASSIFIER
# ============================================================
mlp_model_path = DIRS['xgb'] / 'MLP_ANN_SMCI_Model.keras'
mlp_scaler_path = DIRS['xgb'] / 'MLP_ANN_StandardScaler.joblib'
mlp_perf_path = DIRS['xgb'] / 'MLP_ANN_SMCI_Model_Performance_Metrics.csv'

if mlp_model_path.exists() and mlp_scaler_path.exists() and not FORCE_RETRAIN_MLP:
    mlp_model = keras.models.load_model(mlp_model_path)
    mlp_scaler = load(mlp_scaler_path)
    log_message("Loaded existing MLP-ANN model and scaler from Drive.")
else:
    mlp_scaler = StandardScaler()
    X_train_scaled = mlp_scaler.fit_transform(X_train)
    X_test_scaled = mlp_scaler.transform(X_test)

    mlp_model = keras.Sequential([
        layers.Input(shape=(len(FEATURE_COLUMNS),)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.20),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.10),
        layers.Dense(3, activation='softmax')
    ])

    mlp_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=MLP_PATIENCE, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5)
    ]

    log_message("Starting MLP-ANN training...")
    mlp_history = mlp_model.fit(
        X_train_scaled, y_train.values,
        validation_data=(X_test_scaled, y_test.values),
        epochs=MLP_EPOCHS,
        batch_size=MLP_BATCH_SIZE,
        callbacks=cb,
        verbose=1
    )

    mlp_model.save(mlp_model_path)
    dump(mlp_scaler, mlp_scaler_path)

    hist_df = pd.DataFrame(mlp_history.history)
    hist_df.to_csv(DIRS['xgb'] / 'MLP_ANN_Training_History.csv', index=False)

    # Training curves
    curve_png = FIG_SUBDIRS['mlp'] / 'MLP_ANN_Training_Validation_Curves.png'
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.plot(hist_df['accuracy'], label='Training accuracy', linewidth=2.5)
    ax.plot(hist_df['val_accuracy'], label='Validation accuracy', linewidth=2.5)
    ax.set_xlabel('Epoch', fontweight='bold')
    ax.set_ylabel('Accuracy', fontweight='bold')
    ax.set_title('MLP-ANN Training and Validation Accuracy', fontsize=18, fontweight='bold')
    ax.grid(True, alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(curve_png, dpi=300, bbox_inches='tight')
    plt.show()

# Ensure scaled data exists
X_train_scaled = mlp_scaler.transform(X_train)
X_test_scaled = mlp_scaler.transform(X_test)

train_prob_mlp = mlp_model.predict(X_train_scaled, verbose=0)
test_prob_mlp = mlp_model.predict(X_test_scaled, verbose=0)
train_pred_mlp = np.argmax(train_prob_mlp, axis=1)
test_pred_mlp = np.argmax(test_prob_mlp, axis=1)

mlp_performance_df = pd.DataFrame([
    multiclass_performance(y_train, train_pred_mlp, train_prob_mlp, y_train_bin, 'Training', 'MLP-ANN'),
    multiclass_performance(y_test, test_pred_mlp, test_prob_mlp, y_test_bin, 'Testing', 'MLP-ANN')
])
mlp_performance_df.to_csv(mlp_perf_path, index=False)

mlp_report = pd.DataFrame(classification_report(y_test, test_pred_mlp, output_dict=True, zero_division=0)).transpose()
mlp_report.to_csv(DIRS['xgb'] / 'MLP_ANN_SMCI_Testing_Classification_Report.csv')

display(mlp_performance_df)
display(mlp_report)


# Cell 30 — Model Comparison: XGBoost versus MLP-ANN

In [ ]:
# ============================================================
# CELL 30 — MODEL COMPARISON: XGBOOST VERSUS MLP-ANN
# ============================================================
comparison_df = pd.concat([xgb_performance_df, mlp_performance_df], ignore_index=True)
comparison_path = DIRS['final'] / 'XGBoost_vs_MLP_ANN_Model_Comparison.csv'
comparison_df.to_csv(comparison_path, index=False)

test_comparison = comparison_df[comparison_df['Dataset'] == 'Testing'].copy()
comparison_png = DIRS['figures'] / 'XGBoost_vs_MLP_ANN_Testing_Performance.png'
if not comparison_png.exists() or FORCE_REMAKE_FIGURES:
    metrics_to_plot = ['Accuracy', 'Balanced_Accuracy', 'F1_Macro', 'ROC_AUC_OVR_Macro']
    plot_df = test_comparison[['Model'] + metrics_to_plot].melt(id_vars='Model', var_name='Metric', value_name='Value')
    fig, ax = plt.subplots(figsize=(11, 7))
    x_pos = np.arange(len(metrics_to_plot))
    width = 0.35
    models = plot_df['Model'].unique().tolist()
    for i, model in enumerate(models):
        vals = [plot_df[(plot_df['Model'] == model) & (plot_df['Metric'] == m)]['Value'].values[0] for m in metrics_to_plot]
        ax.bar(x_pos + (i - 0.5) * width, vals, width, label=model)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(metrics_to_plot, rotation=20, ha='right', fontweight='bold')
    ax.set_ylabel('Metric value', fontweight='bold')
    ax.set_title('Testing Performance Comparison: XGBoost and MLP-ANN', fontsize=18, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(comparison_png, dpi=300, bbox_inches='tight')
    plt.show()

display(comparison_df)


# Cell 31 — Predict January 2024, January 2030 and January 2050 with Both Models

In [ ]:
# ============================================================
# CELL 31 — PREDICT JANUARY 2024, JANUARY 2030 AND JANUARY 2050 WITH BOTH MODELS
# ============================================================
def prepare_prediction_table(path_or_df, year):
    if isinstance(path_or_df, (str, Path)):
        df = safe_read_csv(path_or_df)
    else:
        df = path_or_df.copy()
    df['OBJECTID'] = df['OBJECTID'].astype(str)
    df = apply_clamps(df, DYNAMIC_FACTORS)
    df, _ = clean_numeric_features(df, FEATURE_COLUMNS, medians=feature_medians)
    df['Prediction_Year'] = year
    return df[['OBJECTID', 'X', 'Y', 'Prediction_Year'] + FEATURE_COLUMNS]

pred_input_paths = {
    2024: jan2024_full_path,
    2030: forecast_2030_path,
    2050: forecast_2050_path
}

prediction_outputs = {}

for year, pth in pred_input_paths.items():
    input_df = prepare_prediction_table(pth, year)
    X_pred = input_df[FEATURE_COLUMNS].copy()

    # XGBoost prediction
    xgb_pred = best_xgb.predict(X_pred)
    xgb_prob = best_xgb.predict_proba(X_pred)
    xgb_out = input_df.copy()
    xgb_out['Predicted_SMCI_Class'] = xgb_pred
    xgb_out['Predicted_SMCI_Class_Label'] = xgb_out['Predicted_SMCI_Class'].map(CLASS_LABELS)
    xgb_out['Probability_Class_0_Low'] = xgb_prob[:, 0]
    xgb_out['Probability_Class_1_Moderate'] = xgb_prob[:, 1]
    xgb_out['Probability_Class_2_Moist'] = xgb_prob[:, 2]
    xgb_out['SMCI_Score'] = (0 * xgb_prob[:, 0] + 1 * xgb_prob[:, 1] + 2 * xgb_prob[:, 2]) / 2.0
    xgb_csv = DIRS['xgb'] / f'XGBoost_SMCI_All_Predictions_{year}.csv'
    xgb_out.to_csv(xgb_csv, index=False)

    # MLP prediction
    X_pred_scaled = mlp_scaler.transform(X_pred)
    mlp_prob = mlp_model.predict(X_pred_scaled, verbose=0)
    mlp_pred = np.argmax(mlp_prob, axis=1)
    mlp_out = input_df.copy()
    mlp_out['Predicted_SMCI_Class'] = mlp_pred
    mlp_out['Predicted_SMCI_Class_Label'] = mlp_out['Predicted_SMCI_Class'].map(CLASS_LABELS)
    mlp_out['Probability_Class_0_Low'] = mlp_prob[:, 0]
    mlp_out['Probability_Class_1_Moderate'] = mlp_prob[:, 1]
    mlp_out['Probability_Class_2_Moist'] = mlp_prob[:, 2]
    mlp_out['SMCI_Score'] = (0 * mlp_prob[:, 0] + 1 * mlp_prob[:, 1] + 2 * mlp_prob[:, 2]) / 2.0
    mlp_csv = DIRS['xgb'] / f'MLP_ANN_SMCI_All_Predictions_{year}.csv'
    mlp_out.to_csv(mlp_csv, index=False)

    prediction_outputs[('XGBoost', year)] = xgb_out
    prediction_outputs[('MLP_ANN', year)] = mlp_out
    print(f"Saved predictions for {year}: XGBoost and MLP-ANN")

print("Prediction outputs prepared.")


# Cell 32 — Load STMA Boundary and Build 30 m Raster Grid Template

In [ ]:
# ============================================================
# CELL 32 — LOAD STMA BOUNDARY AND BUILD 30 m RASTER GRID TEMPLATE
# ============================================================
# This local rasterisation step converts 30 m gridded points to GeoTIFFs.
# It uses UTM Zone 30N and clips to the STMA boundary where possible.

boundary_geojson_path = DIRS['metadata'] / 'STMA_Boundary_GeoJSON.json'

try:
    if boundary_geojson_path.exists():
        with open(boundary_geojson_path, 'r', encoding='utf-8') as f:
            boundary_info = json.load(f)
    else:
        boundary_info = stma_boundary.geometry().getInfo()
        with open(boundary_geojson_path, 'w', encoding='utf-8') as f:
            json.dump(boundary_info, f)
    boundary_geom_lonlat = shape(boundary_info)
    transformer_geom = Transformer.from_crs('EPSG:4326', OUTPUT_CRS, always_xy=True)
    boundary_geom_utm = shapely_transform(transformer_geom.transform, boundary_geom_lonlat)
    boundary_shapes_utm = [mapping(boundary_geom_utm)]
    print("STMA boundary loaded for clipping.")
except Exception as e:
    boundary_geom_utm = None
    boundary_shapes_utm = None
    log_error("Boundary_Load", e)
    print("Boundary clipping will be skipped if boundary loading failed.")

# Build template from January 2024 full grid points.
template_df = jan2024_full_df[['OBJECTID', 'X', 'Y']].copy()
template_df['X'] = pd.to_numeric(template_df['X'], errors='coerce')  # latitude
template_df['Y'] = pd.to_numeric(template_df['Y'], errors='coerce')  # longitude
template_df = template_df.dropna(subset=['X', 'Y']).copy()

transformer_points = Transformer.from_crs('EPSG:4326', OUTPUT_CRS, always_xy=True)
xs_utm, ys_utm = transformer_points.transform(template_df['Y'].values, template_df['X'].values)
template_df['x_utm'] = xs_utm
template_df['y_utm'] = ys_utm

origin_x = math.floor((template_df['x_utm'].min() - CELL_SIZE_M / 2) / CELL_SIZE_M) * CELL_SIZE_M
origin_y = math.ceil((template_df['y_utm'].max() + CELL_SIZE_M / 2) / CELL_SIZE_M) * CELL_SIZE_M
max_x = math.ceil((template_df['x_utm'].max() + CELL_SIZE_M / 2) / CELL_SIZE_M) * CELL_SIZE_M
min_y = math.floor((template_df['y_utm'].min() - CELL_SIZE_M / 2) / CELL_SIZE_M) * CELL_SIZE_M

width = int(math.ceil((max_x - origin_x) / CELL_SIZE_M))
height = int(math.ceil((origin_y - min_y) / CELL_SIZE_M))
transform = from_origin(origin_x, origin_y, CELL_SIZE_M, CELL_SIZE_M)

template_df['col'] = np.floor((template_df['x_utm'] - origin_x) / CELL_SIZE_M).astype(int)
template_df['row'] = np.floor((origin_y - template_df['y_utm']) / CELL_SIZE_M).astype(int)
template_df = template_df[(template_df['row'] >= 0) & (template_df['row'] < height) & (template_df['col'] >= 0) & (template_df['col'] < width)].copy()

grid_template_path = DIRS['metadata'] / 'Raster_Grid_Template.csv'
template_df[['OBJECTID', 'X', 'Y', 'x_utm', 'y_utm', 'row', 'col']].to_csv(grid_template_path, index=False)

print("Raster template created.")
print("Width:", width, "Height:", height, "Cell size:", CELL_SIZE_M)
print("Template points:", template_df.shape[0])


# Cell 33 — Rasterisation, Natural Breaks and Percentage Functions

In [ ]:
# ============================================================
# CELL 33 — RASTERISATION, NATURAL BREAKS AND PERCENTAGE FUNCTIONS
# ============================================================
NODATA_FLOAT = -9999.0
NODATA_UINT = 0


def df_to_raster_array(df, value_col, dtype='float32', nodata=NODATA_FLOAT, reducer='mean'):
    df = df.copy()
    df['OBJECTID'] = df['OBJECTID'].astype(str)
    merged = template_df[['OBJECTID', 'row', 'col']].merge(df[['OBJECTID', value_col]], on='OBJECTID', how='left')
    merged[value_col] = pd.to_numeric(merged[value_col], errors='coerce')
    merged = merged.dropna(subset=[value_col, 'row', 'col']).copy()

    if reducer == 'mean':
        grouped = merged.groupby(['row', 'col'])[value_col].mean().reset_index()
    else:
        grouped = merged.groupby(['row', 'col'])[value_col].first().reset_index()

    arr = np.full((height, width), nodata, dtype=dtype)
    rows = grouped['row'].astype(int).values
    cols = grouped['col'].astype(int).values
    vals = grouped[value_col].values
    arr[rows, cols] = vals.astype(dtype)
    return arr


def save_geotiff(array, out_path, dtype='float32', nodata=NODATA_FLOAT, clip=True):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    profile = {
        'driver': 'GTiff',
        'height': array.shape[0],
        'width': array.shape[1],
        'count': 1,
        'dtype': dtype,
        'crs': OUTPUT_CRS,
        'transform': transform,
        'nodata': nodata,
        'compress': 'lzw'
    }
    temp_path = out_path.with_suffix('.tmp.tif')
    with rasterio.open(temp_path, 'w', **profile) as dst:
        dst.write(array, 1)

    if clip and boundary_shapes_utm is not None:
        try:
            with rasterio.open(temp_path) as src:
                clipped, clipped_transform = rio_mask(src, boundary_shapes_utm, crop=True, nodata=nodata)
                clipped_profile = src.profile.copy()
                clipped_profile.update({
                    'height': clipped.shape[1],
                    'width': clipped.shape[2],
                    'transform': clipped_transform,
                    'compress': 'lzw'
                })
            with rasterio.open(out_path, 'w', **clipped_profile) as dst:
                dst.write(clipped)
            temp_path.unlink(missing_ok=True)
        except Exception as e:
            log_error(f"Raster_Clip_{out_path.name}", e)
            shutil.move(str(temp_path), str(out_path))
    else:
        shutil.move(str(temp_path), str(out_path))


def natural_breaks_classify(score_array, n_classes=5):
    valid = score_array[(score_array != NODATA_FLOAT) & np.isfinite(score_array)]
    if valid.size == 0:
        return np.full(score_array.shape, NODATA_UINT, dtype='uint8'), []

    try:
        if len(np.unique(valid)) < n_classes:
            raise ValueError('Not enough unique values for Jenks breaks')
        breaks = jenkspy.jenks_breaks(valid, n_classes=n_classes)
    except Exception:
        breaks = np.quantile(valid, np.linspace(0, 1, n_classes + 1)).tolist()
        breaks[0] = float(np.min(valid))
        breaks[-1] = float(np.max(valid))

    class_arr = np.full(score_array.shape, NODATA_UINT, dtype='uint8')
    valid_mask = (score_array != NODATA_FLOAT) & np.isfinite(score_array)
    for i in range(n_classes):
        lo, hi = breaks[i], breaks[i + 1]
        if i == 0:
            mask_i = valid_mask & (score_array >= lo) & (score_array <= hi)
        elif i == n_classes - 1:
            mask_i = valid_mask & (score_array > lo) & (score_array <= hi)
        else:
            mask_i = valid_mask & (score_array > lo) & (score_array <= hi)
        class_arr[mask_i] = i + 1
    return class_arr, breaks


def class_area_percentages(class_arr, model_name, year, breaks):
    valid_mask = class_arr != NODATA_UINT
    total_pixels = int(valid_mask.sum())
    records = []
    for class_id in range(1, 6):
        count = int((class_arr == class_id).sum())
        area_m2 = count * CELL_SIZE_M * CELL_SIZE_M
        area_ha = area_m2 / 10000.0
        pct = (count / total_pixels * 100) if total_pixels > 0 else np.nan
        records.append({
            'Model': model_name,
            'Year': year,
            'Class_ID': class_id,
            'Class_Label': NB_CLASS_LABELS[class_id],
            'Pixel_Count': count,
            'Area_m2': area_m2,
            'Area_ha': area_ha,
            'Percentage': pct,
            'Jenks_Lower_Break': breaks[class_id - 1] if len(breaks) == 6 else np.nan,
            'Jenks_Upper_Break': breaks[class_id] if len(breaks) == 6 else np.nan
        })
    return pd.DataFrame(records)

log_message("Rasterisation and natural-break functions loaded.")


# Cell 34 — Export 10 Factor Layers for January 2024, January 2030 and January 2050

In [ ]:
# ============================================================
# CELL 34 — EXPORT 10 FACTOR LAYERS FOR JANUARY 2024, JANUARY 2030 AND JANUARY 2050
# ============================================================
factor_tables = {
    2024: safe_read_csv(jan2024_full_path),
    2030: safe_read_csv(forecast_2030_path),
    2050: safe_read_csv(forecast_2050_path)
}

for year, df in factor_tables.items():
    df['OBJECTID'] = df['OBJECTID'].astype(str)
    df = apply_clamps(df, DYNAMIC_FACTORS)
    df, _ = clean_numeric_features(df, FEATURE_COLUMNS, medians=feature_medians)

    if year == 2024:
        out_dir = DIRS['layers_2024']
    elif year == 2030:
        out_dir = DIRS['layers_2030']
    else:
        out_dir = DIRS['layers_2050']

    for factor in FEATURE_COLUMNS:
        out_tif = out_dir / f'{factor}_January{year}_30m.tif'
        if out_tif.exists() and not FORCE_RERASTERIZE:
            continue
        arr = df_to_raster_array(df, factor, dtype='float32', nodata=NODATA_FLOAT, reducer='mean')
        save_geotiff(arr, out_tif, dtype='float32', nodata=NODATA_FLOAT, clip=True)
        print("Saved factor layer:", out_tif.name)

print("All requested 10-factor layers were created or already existed.")


# Cell 35 — Export Three-Class, Probability, Score and Natural-Break Predictive Maps

In [ ]:
# ============================================================
# CELL 35 — EXPORT THREE-CLASS, PROBABILITY, SCORE AND NATURAL-BREAK PREDICTIVE MAPS
# ============================================================
all_percentage_tables = []

for (model_name, year), df in prediction_outputs.items():
    safe_model = model_name.replace('-', '_')
    df['OBJECTID'] = df['OBJECTID'].astype(str)

    # Three-class map
    class_tif = FIG_SUBDIRS['three_class'] / f'{safe_model}_Three_Class_SMCI_January{year}_30m.tif'
    if not class_tif.exists() or FORCE_RERASTERIZE:
        arr_class = df_to_raster_array(df, 'Predicted_SMCI_Class', dtype='float32', nodata=NODATA_FLOAT, reducer='first')
        save_geotiff(arr_class.astype('float32'), class_tif, dtype='float32', nodata=NODATA_FLOAT, clip=True)
        print("Saved three-class map:", class_tif.name)

    # Probability maps
    for prob_col in ['Probability_Class_0_Low', 'Probability_Class_1_Moderate', 'Probability_Class_2_Moist']:
        prob_tif = FIG_SUBDIRS['probability'] / f'{safe_model}_{prob_col}_January{year}_30m.tif'
        if not prob_tif.exists() or FORCE_RERASTERIZE:
            prob_arr = df_to_raster_array(df, prob_col, dtype='float32', nodata=NODATA_FLOAT, reducer='mean')
            save_geotiff(prob_arr, prob_tif, dtype='float32', nodata=NODATA_FLOAT, clip=True)
            print("Saved probability map:", prob_tif.name)

    # Continuous score map
    score_tif = FIG_SUBDIRS['scores'] / f'{safe_model}_SMCI_Score_January{year}_30m.tif'
    score_arr = df_to_raster_array(df, 'SMCI_Score', dtype='float32', nodata=NODATA_FLOAT, reducer='mean')
    if not score_tif.exists() or FORCE_RERASTERIZE:
        save_geotiff(score_arr, score_tif, dtype='float32', nodata=NODATA_FLOAT, clip=True)
        print("Saved score map:", score_tif.name)

    # Natural breaks 5-class map
    nb_tif = FIG_SUBDIRS['natural_breaks'] / f'{safe_model}_SMCI_NaturalBreaks_5Class_January{year}_30m.tif'
    nb_class_arr, breaks = natural_breaks_classify(score_arr, n_classes=5)
    if not nb_tif.exists() or FORCE_RERASTERIZE:
        save_geotiff(nb_class_arr, nb_tif, dtype='uint8', nodata=NODATA_UINT, clip=True)
        print("Saved natural-break map:", nb_tif.name)

    pct_df = class_area_percentages(nb_class_arr, model_name, year, breaks)
    pct_csv = FIG_SUBDIRS['percentages'] / f'{safe_model}_SMCI_Class_Percentages_January{year}.csv'
    pct_df.to_csv(pct_csv, index=False)
    all_percentage_tables.append(pct_df)

combined_percentage_df = pd.concat(all_percentage_tables, ignore_index=True)
combined_percentage_path = FIG_SUBDIRS['percentages'] / 'Combined_Model_Class_Percentage_Comparison.csv'
combined_percentage_df.to_csv(combined_percentage_path, index=False)

display(combined_percentage_df)
print("Natural-break classified maps and percentage tables completed.")


# Cell 36 — Create Final Excel Summary Workbooks

In [ ]:
# ============================================================
# CELL 36 — CREATE FINAL EXCEL SUMMARY WORKBOOKS
# ============================================================
summary_xlsx = DIRS['final'] / 'STMA_SMCI_Final_Results.xlsx'
explain_xlsx = DIRS['final'] / 'STMA_SMCI_Explainability_and_Model_Outputs.xlsx'

with pd.ExcelWriter(summary_xlsx, engine='openpyxl') as writer:
    comparison_df.to_excel(writer, sheet_name='Model Comparison', index=False)
    combined_percentage_df.to_excel(writer, sheet_name='Class Area Percentages', index=False)
    if point_metrics_path.exists():
        pd.read_csv(point_metrics_path).head(200000).to_excel(writer, sheet_name='Prophet Point Metrics', index=False)
    if factor_metrics_path.exists():
        pd.read_csv(factor_metrics_path).to_excel(writer, sheet_name='Prophet Factor Summary', index=False)
    if overall_metrics_path.exists():
        pd.read_csv(overall_metrics_path).to_excel(writer, sheet_name='Prophet Overall Metrics', index=False)
    split_report.to_excel(writer, sheet_name='Train Test Split', index=False)

with pd.ExcelWriter(explain_xlsx, engine='openpyxl') as writer:
    pearson_corr.to_excel(writer, sheet_name='Pearson Correlation')
    cv_diagnostic_df.to_excel(writer, sheet_name='XGB 5Fold CV', index=False)
    cv_summary_df.to_excel(writer, sheet_name='XGB CV Summary', index=False)
    feature_importance_df.to_excel(writer, sheet_name='XGB Feature Importance', index=False)
    shap_importance_df.to_excel(writer, sheet_name='SHAP Importance', index=False)
    if interaction_csv.exists():
        pd.read_csv(interaction_csv, index_col=0).to_excel(writer, sheet_name='SHAP Interaction')
    xgb_performance_df.to_excel(writer, sheet_name='XGB Performance', index=False)
    mlp_performance_df.to_excel(writer, sheet_name='MLP Performance', index=False)

print("Final summary workbooks saved:")
print(summary_xlsx)
print(explain_xlsx)


# Cell 37 — Zip Final Outputs

In [ ]:
# ============================================================
# CELL 37 — ZIP FINAL OUTPUTS
# ============================================================
# This creates final ZIP archives. Large folders may take time.

def make_zip(source_dir, zip_name):
    source_dir = Path(source_dir)
    zip_base = DIRS['final'] / zip_name
    zip_path = shutil.make_archive(str(zip_base), 'zip', source_dir)
    print("Created ZIP:", zip_path)
    return zip_path

# Separate archives keep download sizes more manageable.
make_zip(DIRS['figures'], 'STMA_SMCI_Explainability_Figures')
make_zip(DIRS['prediction_layers'], 'STMA_SMCI_Predictive_Maps_and_Percentages')
make_zip(DIRS['forecast_tables'], 'STMA_SMCI_Forecast_Point_Tables')
make_zip(DIRS['final'], 'STMA_SMCI_Final_Summaries')

log_message("Final ZIP files created.")


# Cell 38 — Final Completion Check

In [ ]:
# ============================================================
# CELL 38 — FINAL COMPLETION CHECK
# ============================================================
check_items = {
    'Combined historical table': combined_path,
    'Prophet 2030 forecast table': forecast_2030_path,
    'Prophet 2050 forecast table': forecast_2050_path,
    'January 2024 full-grid table': jan2024_full_path,
    'XGBoost model': xgb_model_path,
    'MLP-ANN model': mlp_model_path,
    'Model comparison': comparison_path,
    'Combined class percentage table': combined_percentage_path,
    'Final summary workbook': summary_xlsx,
    'Explainability workbook': explain_xlsx
}

completion_records = []
for name, path in check_items.items():
    p = Path(path)
    completion_records.append({
        'Item': name,
        'Path': str(p),
        'Exists': p.exists(),
        'Size_MB': round(p.stat().st_size / (1024 * 1024), 3) if p.exists() else 0
    })
completion_df = pd.DataFrame(completion_records)
completion_df.to_csv(DIRS['metadata'] / 'Final_Completion_Check.csv', index=False)

display(completion_df)
log_message("Notebook completed. Review missing items if any Exists=False.")
